In [0]:
print("Spark_Version:", spark.version)
test_df = spark.range(1,6)
test_df.display()

Spark_Version: 4.2.0


id
1
2
3
4
5


In [0]:
# ============================================
# HEALTHCARE PATIENT FLOW PROJECT
# STEP 1: SOURCE CONFIGURATION
# ============================================

source_path = "/Volumes/workspace/default/healthcare_source_data"

source_files = {
    "patients": f"{source_path}/patients.csv",
    "encounters": f"{source_path}/encounters.csv",
    "conditions": f"{source_path}/conditions.csv",
    "procedures": f"{source_path}/procedures.csv"
}

print("Source location configured successfully.")

for name, path in source_files.items():
    print(f"{name}: {path}")

Source location configured successfully.
patients: /Volumes/workspace/default/healthcare_source_data/patients.csv
encounters: /Volumes/workspace/default/healthcare_source_data/encounters.csv
conditions: /Volumes/workspace/default/healthcare_source_data/conditions.csv
procedures: /Volumes/workspace/default/healthcare_source_data/procedures.csv


In [0]:
# ============================================
# STEP 2: READ SOURCE FILES INTO SPARK
# ============================================

from pyspark.sql import functions as F

source_data = {}

for dataset_name, file_path in source_files.items():

    df = (
        spark.read
        .format("csv")
        .option("header", "true")
        .option("inferSchema", "true")
        .option("mode", "PERMISSIVE")
        .load(file_path)
    )

    source_data[dataset_name] = df

    print(f"Loaded DataFrame: {dataset_name}")

print("All source DataFrames initialized.")

Loaded DataFrame: patients
Loaded DataFrame: encounters
Loaded DataFrame: conditions
Loaded DataFrame: procedures
All source DataFrames initialized.


In [0]:
# ============================================
# STEP 3: DATASET PROFILING
# ============================================

dataset_summary = []

for dataset_name, df in source_data.items():

    print("=" * 80)
    print(f"DATASET: {dataset_name.upper()}")
    print("=" * 80)

    # Record count
    record_count = df.count()

    # Column count
    column_count = len(df.columns)

    print(f"Total Records: {record_count}")
    print(f"Total Columns: {column_count}")

    # Print schema and data types
    print("\nSchema:")
    df.printSchema()

    # Display sample records
    print("\nSample records:")
    display(df.limit(5))

    dataset_summary.append(
        (dataset_name, record_count, column_count)
    )

print("\nPROFILE SUMMARY")

for name, records, columns in dataset_summary:
    print(
        f"Dataset: {name} | "
        f"Records: {records} | "
        f"Columns: {columns}"
    )

DATASET: PATIENTS
Total Records: 1163
Total Columns: 25

Schema:
root
 |-- Id: string (nullable = true)
 |-- BIRTHDATE: date (nullable = true)
 |-- DEATHDATE: date (nullable = true)
 |-- SSN: string (nullable = true)
 |-- DRIVERS: string (nullable = true)
 |-- PASSPORT: string (nullable = true)
 |-- PREFIX: string (nullable = true)
 |-- FIRST: string (nullable = true)
 |-- LAST: string (nullable = true)
 |-- SUFFIX: string (nullable = true)
 |-- MAIDEN: string (nullable = true)
 |-- MARITAL: string (nullable = true)
 |-- RACE: string (nullable = true)
 |-- ETHNICITY: string (nullable = true)
 |-- GENDER: string (nullable = true)
 |-- BIRTHPLACE: string (nullable = true)
 |-- ADDRESS: string (nullable = true)
 |-- CITY: string (nullable = true)
 |-- STATE: string (nullable = true)
 |-- COUNTY: string (nullable = true)
 |-- ZIP: integer (nullable = true)
 |-- LAT: double (nullable = true)
 |-- LON: double (nullable = true)
 |-- HEALTHCARE_EXPENSES: double (nullable = true)
 |-- HEALTHCAR

Id,BIRTHDATE,DEATHDATE,SSN,DRIVERS,PASSPORT,PREFIX,FIRST,LAST,SUFFIX,MAIDEN,MARITAL,RACE,ETHNICITY,GENDER,BIRTHPLACE,ADDRESS,CITY,STATE,COUNTY,ZIP,LAT,LON,HEALTHCARE_EXPENSES,HEALTHCARE_COVERAGE
b9c610cd-28a6-4636-ccb6-c7a0d2a4cb85,2019-02-17,null,999-65-3251,null,null,null,Damon455,Langosh790,null,null,null,white,nonhispanic,M,Middleborough Massachusetts US,620 Lynch Tunnel Apt 0,Springfield,Massachusetts,Hampden County,1104,42.08038942501558,-72.48043144917739,9039.1645,7964.1255
c1f1fcaa-82fd-d5b7-3544-c8f9708b06a8,2005-07-04,null,999-49-3323,S99941126,null,null,Thi53,Wunsch504,null,null,null,white,nonhispanic,F,Danvers Massachusetts US,972 Tillman Branch Suite 48,Bellingham,Massachusetts,Norfolk County,null,42.03521335752818,-71.48251904737748,402723.415,14064.135000000002
339144f8-50e1-633e-a013-f361391c4cff,1998-05-11,null,999-10-8743,S99996708,X75063318X,Mr.,Chi716,Greenfelder433,null,null,null,white,nonhispanic,M,Athens Athens Prefecture GR,1060 Bernhard Crossroad Suite 15,Boston,Massachusetts,Suffolk County,2131,42.29255662362827,-71.06116042204106,571935.8725,787.5374999999999
d488232e-bf14-4bed-08c0-a82f34b6a197,2003-01-28,null,999-56-6057,S99929424,null,Ms.,Phillis443,Walter473,null,null,null,white,nonhispanic,F,Boston Massachusetts US,677 Ritchie Terrace,Hingham,Massachusetts,Plymouth County,2043,42.200490824321214,-70.91607557520302,582557.803,104782.20699999998
217f95a3-4e10-bd5d-fb67-0cfb5e8ba075,1993-12-23,null,999-91-4320,S99991143,X44132498X,Mr.,Jerrold404,Herzog843,null,null,M,black,nonhispanic,M,Boston Massachusetts US,276 Bernier Branch,Revere,Massachusetts,Suffolk County,null,42.3818748756275,-70.99928578468784,475826.855,18067.094999999998


DATASET: ENCOUNTERS
Total Records: 61459
Total Columns: 15

Schema:
root
 |-- Id: string (nullable = true)
 |-- START: timestamp (nullable = true)
 |-- STOP: timestamp (nullable = true)
 |-- PATIENT: string (nullable = true)
 |-- ORGANIZATION: string (nullable = true)
 |-- PROVIDER: string (nullable = true)
 |-- PAYER: string (nullable = true)
 |-- ENCOUNTERCLASS: string (nullable = true)
 |-- CODE: integer (nullable = true)
 |-- DESCRIPTION: string (nullable = true)
 |-- BASE_ENCOUNTER_COST: double (nullable = true)
 |-- TOTAL_CLAIM_COST: double (nullable = true)
 |-- PAYER_COVERAGE: double (nullable = true)
 |-- REASONCODE: long (nullable = true)
 |-- REASONDESCRIPTION: string (nullable = true)


Sample records:


Id,START,STOP,PATIENT,ORGANIZATION,PROVIDER,PAYER,ENCOUNTERCLASS,CODE,DESCRIPTION,BASE_ENCOUNTER_COST,TOTAL_CLAIM_COST,PAYER_COVERAGE,REASONCODE,REASONDESCRIPTION
748f8357-6cc7-551d-f31a-32fa2cf84126,2019-02-17T05:07:38.000Z,2019-02-17T05:22:38.000Z,b9c610cd-28a6-4636-ccb6-c7a0d2a4cb85,f7ae497d-8dc6-3721-9402-43b621a4e7d2,82608ebb-037c-3cef-9d34-3736d69b29e8,7c4411ce-02f1-39b5-b9ec-dfbea9ad3c1a,wellness,410620009,Well child visit (procedure),129.16,877.79,833.9,null,null
5a4735ae-423f-6563-28ab-b3d11b49b2d4,2019-03-24T05:07:38.000Z,2019-03-24T05:22:38.000Z,b9c610cd-28a6-4636-ccb6-c7a0d2a4cb85,f7ae497d-8dc6-3721-9402-43b621a4e7d2,82608ebb-037c-3cef-9d34-3736d69b29e8,7c4411ce-02f1-39b5-b9ec-dfbea9ad3c1a,wellness,410620009,Well child visit (procedure),129.16,269.68,256.2,null,null
0bee1ce6-3e2c-5506-f71c-a7ba8f64a3d3,2019-05-26T05:07:38.000Z,2019-05-26T05:22:38.000Z,b9c610cd-28a6-4636-ccb6-c7a0d2a4cb85,f7ae497d-8dc6-3721-9402-43b621a4e7d2,82608ebb-037c-3cef-9d34-3736d69b29e8,7c4411ce-02f1-39b5-b9ec-dfbea9ad3c1a,wellness,410620009,Well child visit (procedure),129.16,1292.75,1228.11,null,null
6e93bcf9-45a4-8528-0120-1c1eaa930faf,2019-07-28T05:07:38.000Z,2019-07-28T05:22:38.000Z,b9c610cd-28a6-4636-ccb6-c7a0d2a4cb85,f7ae497d-8dc6-3721-9402-43b621a4e7d2,82608ebb-037c-3cef-9d34-3736d69b29e8,7c4411ce-02f1-39b5-b9ec-dfbea9ad3c1a,wellness,410620009,Well child visit (procedure),129.16,1323.87,1257.68,null,null
8b6787c3-4316-a0cb-899d-4746525c319f,2019-10-27T05:07:38.000Z,2019-10-27T05:22:38.000Z,b9c610cd-28a6-4636-ccb6-c7a0d2a4cb85,f7ae497d-8dc6-3721-9402-43b621a4e7d2,82608ebb-037c-3cef-9d34-3736d69b29e8,7c4411ce-02f1-39b5-b9ec-dfbea9ad3c1a,wellness,410620009,Well child visit (procedure),129.16,831.76,790.17,null,null


DATASET: CONDITIONS
Total Records: 38094
Total Columns: 6

Schema:
root
 |-- START: date (nullable = true)
 |-- STOP: date (nullable = true)
 |-- PATIENT: string (nullable = true)
 |-- ENCOUNTER: string (nullable = true)
 |-- CODE: long (nullable = true)
 |-- DESCRIPTION: string (nullable = true)


Sample records:


START,STOP,PATIENT,ENCOUNTER,CODE,DESCRIPTION
2013-06-24,2013-07-02,c1f1fcaa-82fd-d5b7-3544-c8f9708b06a8,0b2794bd-ec2b-d34f-0610-2523b3b7fcf0,10509002,Acute bronchitis (disorder)
2016-02-27,2016-03-14,c1f1fcaa-82fd-d5b7-3544-c8f9708b06a8,a6d818dd-0983-fd1c-eefa-3d2295532c45,283371005,Laceration of forearm
2016-08-11,2016-08-22,c1f1fcaa-82fd-d5b7-3544-c8f9708b06a8,36d2e781-4655-0a11-1f70-c69856e02019,444814009,Viral sinusitis (disorder)
2016-11-27,2016-12-17,c1f1fcaa-82fd-d5b7-3544-c8f9708b06a8,c8eaaf41-958b-31ab-7de5-568cee8751f3,444814009,Viral sinusitis (disorder)
2017-02-22,2017-06-02,c1f1fcaa-82fd-d5b7-3544-c8f9708b06a8,6474f606-5a1b-48c0-bbbf-ad6dcbc24d4e,16114001,Fracture of ankle


DATASET: PROCEDURES
Total Records: 83823
Total Columns: 9

Schema:
root
 |-- START: timestamp (nullable = true)
 |-- STOP: timestamp (nullable = true)
 |-- PATIENT: string (nullable = true)
 |-- ENCOUNTER: string (nullable = true)
 |-- CODE: long (nullable = true)
 |-- DESCRIPTION: string (nullable = true)
 |-- BASE_COST: double (nullable = true)
 |-- REASONCODE: integer (nullable = true)
 |-- REASONDESCRIPTION: string (nullable = true)


Sample records:


START,STOP,PATIENT,ENCOUNTER,CODE,DESCRIPTION,BASE_COST,REASONCODE,REASONDESCRIPTION
2019-02-17T05:07:38.000Z,2019-02-17T05:22:38.000Z,b9c610cd-28a6-4636-ccb6-c7a0d2a4cb85,748f8357-6cc7-551d-f31a-32fa2cf84126,430193006,Medication Reconciliation (procedure),608.11,null,null
2019-05-26T05:07:38.000Z,2019-05-26T05:22:38.000Z,b9c610cd-28a6-4636-ccb6-c7a0d2a4cb85,0bee1ce6-3e2c-5506-f71c-a7ba8f64a3d3,430193006,Medication Reconciliation (procedure),460.99,null,null
2019-07-28T05:07:38.000Z,2019-07-28T05:22:38.000Z,b9c610cd-28a6-4636-ccb6-c7a0d2a4cb85,6e93bcf9-45a4-8528-0120-1c1eaa930faf,430193006,Medication Reconciliation (procedure),492.11,null,null
2020-02-17T10:07:38.000Z,2020-02-17T10:40:32.000Z,b9c610cd-28a6-4636-ccb6-c7a0d2a4cb85,01efcc52-15d6-51e9-faa2-bee069fcbe44,395142003,Allergy screening test,516.65,null,null
2020-04-26T05:07:38.000Z,2020-04-26T05:22:38.000Z,b9c610cd-28a6-4636-ccb6-c7a0d2a4cb85,1a7debfc-9582-7f23-a109-4f154a182ee2,430193006,Medication Reconciliation (procedure),783.2,null,null



PROFILE SUMMARY
Dataset: patients | Records: 1163 | Columns: 25
Dataset: encounters | Records: 61459 | Columns: 15
Dataset: conditions | Records: 38094 | Columns: 6
Dataset: procedures | Records: 83823 | Columns: 9


In [0]:

'''
We need to answer:
- How many records are there?
- Are any columns completely empty?
- Which columns have missing values?
- Are there duplicate records?
- Are the primary identifiers unique?
- Are patient IDs referenced in encounters valid?
- Are encounter IDs referenced in conditions and procedures valid?

'''
from pyspark.sql import functions as F

# STEP 4: SOURCE DATA PROFILING

for dataset_name, df in source_data.items():

    print("=" * 70)
    print(f"DATASET: {dataset_name.upper()}")
    print("=" * 70)

    # Total rows
    total_records = df.count()

    # Total columns
    total_columns = len(df.columns)

    # Fully duplicated rows
    duplicate_records = total_records - df.dropDuplicates().count()

    print(f"Total Records   : {total_records}")
    print(f"Total Columns   : {total_columns}")
    print(f"Duplicate Rows  : {duplicate_records}")

    # Null profiling
    null_profile = df.select([
        F.count(
            F.when(F.col(c).isNull(), 1)
        ).alias(c)
        for c in df.columns
    ])

    null_profile.show(truncate=False) # -- so that the o/p values are not shortened

    print("Column names:", df.columns)
    print()

DATASET: PATIENTS
Total Records   : 1163
Total Columns   : 25
Duplicate Rows  : 0
+---+---------+---------+---+-------+--------+------+-----+----+------+------+-------+----+---------+------+----------+-------+----+-----+------+---+---+---+-------------------+-------------------+
|Id |BIRTHDATE|DEATHDATE|SSN|DRIVERS|PASSPORT|PREFIX|FIRST|LAST|SUFFIX|MAIDEN|MARITAL|RACE|ETHNICITY|GENDER|BIRTHPLACE|ADDRESS|CITY|STATE|COUNTY|ZIP|LAT|LON|HEALTHCARE_EXPENSES|HEALTHCARE_COVERAGE|
+---+---------+---------+---+-------+--------+------+-----+----+------+------+-------+----+---------+------+----------+-------+----+-----+------+---+---+---+-------------------+-------------------+
|0  |0        |1000     |0  |215    |276     |245   |0    |0   |1147  |832   |384    |0   |0        |0     |0         |0      |0   |0    |0     |545|0  |0  |0                  |0                  |
+---+---------+---------+---+-------+--------+------+-----+----+------+------+-------+----+---------+------+----------+-------

In [0]:
# STEP 5: PRIMARY KEY VALIDATION (OPTIMIZED)

from pyspark.sql import functions as F

primary_keys = {
    "patients": "Id",
    "encounters": "Id"
}

for dataset_name, pk_column in primary_keys.items():

    df = source_data[dataset_name]

    print("=" * 60)
    print(f"PRIMARY KEY CHECK: {dataset_name.upper()}")
    print("=" * 60)

    # Calculate total records and null primary keys together
    basic_stats = df.agg(
        F.count("*").alias("total_records"),
        F.sum(
            F.when(F.col(pk_column).isNull(), 1)
             .otherwise(0)
        ).alias("null_pk_count")
    ).first()

    total_records = basic_stats["total_records"]
    null_pk_count = basic_stats["null_pk_count"] or 0

    # Identify duplicate primary key groups
    duplicate_pk_count = (
        df.groupBy(pk_column)
          .count()
          .filter(F.col("count") > 1)
          .count()
    )

    print(f"Total records         : {total_records}")
    print(f"Null primary keys    : {null_pk_count}")
    print(f"Duplicate key groups : {duplicate_pk_count}")

    if null_pk_count == 0 and duplicate_pk_count == 0:
        print("RESULT: PRIMARY KEY VALID")
    else:
        print("RESULT: PRIMARY KEY INVALID")

    print()

PRIMARY KEY CHECK: PATIENTS
Total records         : 1163
Null primary keys    : 0
Duplicate key groups : 0
RESULT: PRIMARY KEY VALID

PRIMARY KEY CHECK: ENCOUNTERS
Total records         : 61459
Null primary keys    : 0
Duplicate key groups : 0
RESULT: PRIMARY KEY VALID



In [0]:
from pyspark.sql import functions as F

composite_keys = {
    "conditions": ["PATIENT", "ENCOUNTER", "CODE", "START"],
    "procedures": ["PATIENT", "ENCOUNTER", "CODE", "START"]
}

for dataset_name, key_columns in composite_keys.items():

    df = source_data[dataset_name]

    total_records = df.count()

    null_key_records = df.filter(
    F.col("PATIENT").isNull() |
    F.col("ENCOUNTER").isNull() |
    F.col("CODE").isNull() |
    F.col("START").isNull()
).count()

    duplicate_key_groups = (
        df.groupBy(key_columns)
        .count()
        .filter(F.col("count") > 1)
        .count()
    )

    print("=" * 60)
    print(f"COMPOSITE KEY CHECK: {dataset_name.upper()}")
    print("=" * 60)
    print(f"Total records          : {total_records}")
    print(f"Null key records      : {null_key_records}")
    print(f"Duplicate key groups  : {duplicate_key_groups}")

    if null_key_records == 0 and duplicate_key_groups == 0:
        print("RESULT: CANDIDATE KEY VALID")
    else:
        print("RESULT: CANDIDATE KEY ISSUES FOUND")

COMPOSITE KEY CHECK: CONDITIONS
Total records          : 38094
Null key records      : 0
Duplicate key groups  : 0
RESULT: CANDIDATE KEY VALID
COMPOSITE KEY CHECK: PROCEDURES
Total records          : 83823
Null key records      : 0
Duplicate key groups  : 1
RESULT: CANDIDATE KEY ISSUES FOUND


In [0]:
#Step 1 — Identify the duplicate procedure

df = source_data["procedures"]

duplicate_keys = (
    df.groupBy("PATIENT", "ENCOUNTER", "CODE", "START")
    .count()
    .filter(F.col("count") > 1)
)

display(duplicate_keys)

PATIENT,ENCOUNTER,CODE,START,count
436a7fef-6640-96ab-2557-77e40e947bc9,5ce53337-efdd-0bd6-c031-92e0d7b6f2c7,171207006,2021-06-09T12:10:26.000Z,2


In [0]:
#Step 2 — View the actual duplicate records
duplicate_records = df.join(
    duplicate_keys,
    on=["PATIENT", "ENCOUNTER", "CODE", "START"],
    how="inner"
)

display(duplicate_records)

PATIENT,ENCOUNTER,CODE,START,STOP,DESCRIPTION,BASE_COST,REASONCODE,REASONDESCRIPTION,count
436a7fef-6640-96ab-2557-77e40e947bc9,5ce53337-efdd-0bd6-c031-92e0d7b6f2c7,171207006,2021-06-09T12:10:26.000Z,2021-06-09T12:25:26.000Z,Depression screening,516.65,null,null,2
436a7fef-6640-96ab-2557-77e40e947bc9,5ce53337-efdd-0bd6-c031-92e0d7b6f2c7,171207006,2021-06-09T12:10:26.000Z,2021-06-09T12:21:06.000Z,Depression screening (procedure),516.65,null,null,2


In [0]:
# as the prcoedure has stop date, we will check on adding another condition if we are getting duplicated if not that is a valid record
duplicate_records.groupBy(
    "PATIENT",
    "ENCOUNTER",
    "CODE",
    "START",
    "STOP"
).count().filter(
    F.col("count") > 1
).display()

PATIENT,ENCOUNTER,CODE,START,STOP,count


In [0]:
#Now checking for foreign key validation where the real patient exists in patient and encounter table
patients_df = source_data["patients"]
encounters_df = source_data["encounters"]

invalid_encounters = encounters_df.join(
    patients_df,
    encounters_df["PATIENT"] == patients_df["Id"],
    "left_anti"
)

print("Total encounters:", encounters_df.count())
print("Invalid patient references:", invalid_encounters.count())

Total encounters: 61459
Invalid patient references: 0


In [0]:
#checking whether each encounter in condition table is valid encounter in encounters table
conditions_df = source_data["conditions"]
encounters_df = source_data["encounters"]

invalid_conditions = conditions_df.join(
    encounters_df,
    conditions_df["ENCOUNTER"] == encounters_df["Id"],
    "left_anti"
)

print("Total conditions:", conditions_df.count())
print("Invalid encounter references:", invalid_conditions.count())

Total conditions: 38094
Invalid encounter references: 0


In [0]:
#checking whether each encounter in procedures table is valid encounter in encounters table
procedures_df = source_data["procedures"]
encounters_df = source_data["encounters"]

invalid_conditions = procedures_df.join(
    encounters_df,
    procedures_df["ENCOUNTER"] == encounters_df["Id"],
    "left_anti"
)

print("Total conditions:", procedures_df.count())
print("Invalid encounter references:", invalid_conditions.count())

Total conditions: 83823
Invalid encounter references: 0


In [0]:
##checking whether each patient in procedures table is valid patient in patients table
procedures_df = source_data["procedures"]
patients_df = source_data["patients"]

invalid_procedures = procedures_df.join(
    patients_df,
    procedures_df["PATIENT"] == patients_df["Id"],
    "left_anti"
)

print("Total procedures:", procedures_df.count())
print("Invalid patient references:", invalid_procedures.count())

Total procedures: 83823
Invalid patient references: 0


In [0]:
#checking whether each patient in procedures table is valid patient in patients table
conditions_df = source_data["conditions"]
patients_df = source_data["patients"]

invalid_procedures = conditions_df.join(
    patients_df,
    conditions_df["PATIENT"] == patients_df["Id"],
    "left_anti"
)

print("Total procedures:", conditions_df.count())
print("Invalid patient references:", invalid_procedures.count())

Total procedures: 38094
Invalid patient references: 0


In [0]:
#checking if end_date is < start_dayte for an encounter
encounters_df = source_data["encounters"]

invalid_dates = encounters_df.filter(
    F.col("STOP") < F.col("START")
)

print("Total encounters:", encounters_df.count())
print("Invalid date records:", invalid_dates.count())

Total encounters: 61459
Invalid date records: 0


In [0]:
#checking if start or stop date is missing 
missing_dates = encounters_df.filter(
    F.col("START").isNull() |
    F.col("STOP").isNull()
)

print("Missing date records:", missing_dates.count())

Missing date records: 0


In [0]:
conditions_df = source_data["conditions"]
procedures_df = source_data["procedures"]

# CONDITIONS
invalid_condition_dates = conditions_df.filter(
    F.col("STOP").isNotNull() &
    F.col("START").isNotNull() &
    (F.col("STOP") < F.col("START"))
)

missing_condition_start = conditions_df.filter(
    F.col("START").isNull() 
)

# PROCEDURES
invalid_procedure_dates = procedures_df.filter(
    F.col("STOP").isNotNull() &
    F.col("START").isNotNull() &
    (F.col("STOP") < F.col("START"))
)

missing_procedure_dates = procedures_df.filter(
    F.col("START").isNull() |
    F.col("STOP").isNull()
)

# SUMMARY
print("=" * 55)
print("CONDITIONS & PROCEDURES DATE QUALITY CHECK")
print("=" * 55)

print("\nCONDITIONS")
print("Total records:", conditions_df.count())
print("Invalid date sequences:", invalid_condition_dates.count())
print("Missing START dates:", missing_condition_start.count())
print("Missing STOP dates (ongoing/unknown):", conditions_df.filter(F.col("STOP").isNull()).count())

print("\nPROCEDURES")
print("Total records:", procedures_df.count())
print("Invalid date sequences:", invalid_procedure_dates.count())
print("Missing date records:", missing_procedure_dates.count())

CONDITIONS & PROCEDURES DATE QUALITY CHECK

CONDITIONS
Total records: 38094
Invalid date sequences: 0
Missing START dates: 0
Missing STOP dates (ongoing/unknown): 8169

PROCEDURES
Total records: 83823
Invalid date sequences: 0
Missing date records: 0


In [0]:
'''A healthcare expense cannot be negative.
A procedure cost should not be negative.
A patient's death date cannot be earlier than their birth date.
An encounter claim cost should not be negative.'''

# BUSINESS RULE VALIDATION

patients_df = source_data["patients"]
encounters_df = source_data["encounters"]
procedures_df = source_data["procedures"]

# 1. PATIENTS: Negative financial values
invalid_patient_financials = patients_df.filter(
    (F.col("HEALTHCARE_EXPENSES") < 0) |
    (F.col("HEALTHCARE_COVERAGE") < 0)
)

# 2. PATIENTS: Death date before birth date
invalid_patient_dates = patients_df.filter(
    F.col("DEATHDATE").isNotNull() &
    (F.col("DEATHDATE") < F.col("BIRTHDATE"))
)

# 3. ENCOUNTERS: Negative financial values
invalid_encounter_financials = encounters_df.filter(
    (F.col("BASE_ENCOUNTER_COST") < 0) |
    (F.col("TOTAL_CLAIM_COST") < 0) |
    (F.col("PAYER_COVERAGE") < 0)
)

# 4. PROCEDURES: Negative procedure costs
invalid_procedure_costs = procedures_df.filter(
    F.col("BASE_COST") < 0
)

# SUMMARY
print("=" * 55)
print("BUSINESS RULE VALIDATION")
print("=" * 55)

print("\nPATIENTS")
print("Negative financial records:", invalid_patient_financials.count())
print("Invalid birth/death dates:", invalid_patient_dates.count())

print("\nENCOUNTERS")
print("Negative financial records:", invalid_encounter_financials.count())

print("\nPROCEDURES")
print("Negative procedure costs:", invalid_procedure_costs.count())

BUSINESS RULE VALIDATION

PATIENTS
Negative financial records: 0
Invalid birth/death dates: 0

ENCOUNTERS
Negative financial records: 0

PROCEDURES
Negative procedure costs: 0


In [0]:
'''We'll create one summary table containing:
- Dataset
- Validation performed
- Number of failed records/groups
- Status (PASS / FAIL)'''

from pyspark.sql import functions as F

# Source DataFrames
patients_df = source_data["patients"]
encounters_df = source_data["encounters"]
conditions_df = source_data["conditions"]
procedures_df = source_data["procedures"]

quality_results = []

def add_check(dataset, check_name, failed_count):
    quality_results.append((
        dataset,
        check_name,
        failed_count,
        "PASS" if failed_count == 0 else "FAIL"
    ))

# -------------------------
# 1. PRIMARY / CANDIDATE KEYS
# -------------------------

add_check("Patients", "Null primary keys",
          patients_df.filter(F.col("Id").isNull()).count())

add_check("Patients", "Duplicate primary keys",
          patients_df.groupBy("Id").count().filter(F.col("count") > 1).count())

add_check("Encounters", "Null primary keys",
          encounters_df.filter(F.col("Id").isNull()).count())

add_check("Encounters", "Duplicate primary keys",
          encounters_df.groupBy("Id").count().filter(F.col("count") > 1).count())

add_check("Conditions", "Null candidate keys",
          conditions_df.filter(
              F.col("PATIENT").isNull() | F.col("ENCOUNTER").isNull() |
              F.col("CODE").isNull() | F.col("START").isNull()
          ).count())

add_check("Conditions", "Duplicate candidate keys",
          conditions_df.groupBy("PATIENT", "ENCOUNTER", "CODE", "START")
          .count().filter(F.col("count") > 1).count())

add_check("Procedures", "Null candidate keys",
          procedures_df.filter(
              F.col("PATIENT").isNull() | F.col("ENCOUNTER").isNull() |
              F.col("CODE").isNull() | F.col("START").isNull() |
              F.col("STOP").isNull()
          ).count())

add_check("Procedures", "Duplicate candidate keys",
          procedures_df.groupBy("PATIENT", "ENCOUNTER", "CODE", "START", "STOP")
          .count().filter(F.col("count") > 1).count())

# -------------------------
# 2. FOREIGN KEYS
# -------------------------

add_check("Encounters", "Invalid patient references",
          encounters_df.join(patients_df,
              encounters_df["PATIENT"] == patients_df["Id"],
              "left_anti").count())

add_check("Conditions", "Invalid encounter references",
          conditions_df.join(encounters_df,
              conditions_df["ENCOUNTER"] == encounters_df["Id"],
              "left_anti").count())

add_check("Procedures", "Invalid encounter references",
          procedures_df.join(encounters_df,
              procedures_df["ENCOUNTER"] == encounters_df["Id"],
              "left_anti").count())

add_check("Conditions", "Invalid patient references",
          conditions_df.join(patients_df,
              conditions_df["PATIENT"] == patients_df["Id"],
              "left_anti").count())

add_check("Procedures", "Invalid patient references",
          procedures_df.join(patients_df,
              procedures_df["PATIENT"] == patients_df["Id"],
              "left_anti").count())

# -------------------------
# 3. DATE VALIDATION
# -------------------------

add_check("Encounters", "STOP before START",
          encounters_df.filter(F.col("STOP") < F.col("START")).count())

add_check("Encounters", "Missing START/STOP",
          encounters_df.filter(
              F.col("START").isNull() | F.col("STOP").isNull()
          ).count())

add_check("Conditions", "STOP before START",
          conditions_df.filter(
              F.col("STOP").isNotNull() & F.col("START").isNotNull() &
              (F.col("STOP") < F.col("START"))
          ).count())

add_check("Conditions", "Missing START",
          conditions_df.filter(F.col("START").isNull()).count())

add_check("Procedures", "STOP before START",
          procedures_df.filter(
              F.col("STOP").isNotNull() & F.col("START").isNotNull() &
              (F.col("STOP") < F.col("START"))
          ).count())

add_check("Procedures", "Missing START/STOP",
          procedures_df.filter(
              F.col("START").isNull() | F.col("STOP").isNull()
          ).count())

# -------------------------
# 4. BUSINESS RULES
# -------------------------

add_check("Patients", "Negative financial values",
          patients_df.filter(
              (F.col("HEALTHCARE_EXPENSES") < 0) |
              (F.col("HEALTHCARE_COVERAGE") < 0)
          ).count())

add_check("Patients", "Death before birth",
          patients_df.filter(
              F.col("DEATHDATE").isNotNull() &
              (F.col("DEATHDATE") < F.col("BIRTHDATE"))
          ).count())

add_check("Encounters", "Negative financial values",
          encounters_df.filter(
              (F.col("BASE_ENCOUNTER_COST") < 0) |
              (F.col("TOTAL_CLAIM_COST") < 0) |
              (F.col("PAYER_COVERAGE") < 0)
          ).count())

add_check("Procedures", "Negative procedure costs",
          procedures_df.filter(F.col("BASE_COST") < 0).count())

# -------------------------
# FINAL REPORT
# -------------------------

quality_report = spark.createDataFrame(
    quality_results,
    ["dataset", "validation", "failed_records", "status"]
)

display(quality_report.orderBy("dataset", "validation"))

print("Total checks:", len(quality_results))
print("Passed checks:", sum(1 for r in quality_results if r[3] == "PASS"))
print("Failed checks:", sum(1 for r in quality_results if r[3] == "FAIL"))

dataset,validation,failed_records,status
Conditions,Duplicate candidate keys,0,PASS
Conditions,Invalid encounter references,0,PASS
Conditions,Invalid patient references,0,PASS
Conditions,Missing START,0,PASS
Conditions,Null candidate keys,0,PASS
Conditions,STOP before START,0,PASS
Encounters,Duplicate primary keys,0,PASS
Encounters,Invalid patient references,0,PASS
Encounters,Missing START/STOP,0,PASS
Encounters,Negative financial values,0,PASS


Total checks: 23
Passed checks: 23
Failed checks: 0


In [0]:
#We did the loading of data, validation, checking for all the required constraints now the clean data is ready to be loaded into delta table/delat lake where we will perform activties/queries over data and the layer is kown as ronze layer basically converting the sv data into delta tables.
from pyspark.sql import functions as F

for dataset_name, file_path in source_files.items():

    # Read source CSV without converting source values into data types
    bronze_df = (
        spark.read
        .format("csv")
        .option("header", "true")
        .option("inferSchema", "false")
        .load(file_path)
    )

    # Add ingestion metadata
    bronze_df = (
        bronze_df
        .withColumn("_ingested_at", F.current_timestamp())
        .withColumn("_source_file", F.lit(file_path))
        .withColumn("_source_system", F.lit("Synthea"))
    )

    # Define Bronze table name
    table_name = f"workspace.default.healthcare_bronze_{dataset_name}"

    # Persist as Delta table
    bronze_df.write \
        .format("delta") \
        .mode("overwrite") \
        .saveAsTable(table_name)

    print(f"Bronze table created: {table_name}")

Bronze table created: workspace.default.healthcare_bronze_patients
Bronze table created: workspace.default.healthcare_bronze_encounters
Bronze table created: workspace.default.healthcare_bronze_conditions
Bronze table created: workspace.default.healthcare_bronze_procedures


In [0]:
from pyspark.sql import functions as F

datasets = ["patients", "encounters", "conditions", "procedures"]

bronze_validation = []

for dataset in datasets:

    # Original source DataFrame
    source_df = source_data[dataset]

    # Bronze Delta table
    table_name = f"workspace.default.healthcare_bronze_{dataset}"
    bronze_df = spark.table(table_name)

    # Record counts
    source_count = source_df.count()
    bronze_count = bronze_df.count()

    # Metadata validation
    required_metadata = ["_ingested_at", "_source_file", "_source_system"]
    missing_metadata = [c for c in required_metadata if c not in bronze_df.columns]

    # Validation status
    count_status = "PASS" if source_count == bronze_count else "FAIL"
    metadata_status = "PASS" if not missing_metadata else "FAIL"

    overall_status = (
        "PASS" if count_status == "PASS" and metadata_status == "PASS"
        else "FAIL"
    )

    bronze_validation.append((
        dataset,
        source_count,
        bronze_count,
        count_status,
        str(missing_metadata),
        metadata_status,
        overall_status
    ))

# Convert results into a Spark DataFrame
bronze_report = spark.createDataFrame(
    bronze_validation,
    [
        "dataset",
        "source_records",
        "bronze_records",
        "count_status",
        "missing_metadata_columns",
        "metadata_status",
        "overall_status"
    ]
)

display(bronze_report)

print("Total datasets checked:", len(bronze_validation))
print("Passed:", sum(1 for r in bronze_validation if r[6] == "PASS"))
print("Failed:", sum(1 for r in bronze_validation if r[6] == "FAIL"))

dataset,source_records,bronze_records,count_status,missing_metadata_columns,metadata_status,overall_status
patients,1163,1163,PASS,[],PASS,PASS
encounters,61459,61459,PASS,[],PASS,PASS
conditions,38094,38094,PASS,[],PASS,PASS
procedures,83823,83823,PASS,[],PASS,PASS


Total datasets checked: 4
Passed: 4
Failed: 0


In [0]:
datasets = ["patients", "encounters", "conditions", "procedures"]

for dataset in datasets:

    table_name = f"workspace.default.healthcare_bronze_{dataset}"
    bronze_df = spark.table(table_name)

    print("=" * 70)
    print(f"BRONZE TABLE: {table_name}")
    print("=" * 70)

    # Inspect column names and data types
    bronze_df.printSchema()

    # Inspect sample records
    display(bronze_df.limit(3))

BRONZE TABLE: workspace.default.healthcare_bronze_patients
root
 |-- Id: string (nullable = true)
 |-- BIRTHDATE: string (nullable = true)
 |-- DEATHDATE: string (nullable = true)
 |-- SSN: string (nullable = true)
 |-- DRIVERS: string (nullable = true)
 |-- PASSPORT: string (nullable = true)
 |-- PREFIX: string (nullable = true)
 |-- FIRST: string (nullable = true)
 |-- LAST: string (nullable = true)
 |-- SUFFIX: string (nullable = true)
 |-- MAIDEN: string (nullable = true)
 |-- MARITAL: string (nullable = true)
 |-- RACE: string (nullable = true)
 |-- ETHNICITY: string (nullable = true)
 |-- GENDER: string (nullable = true)
 |-- BIRTHPLACE: string (nullable = true)
 |-- ADDRESS: string (nullable = true)
 |-- CITY: string (nullable = true)
 |-- STATE: string (nullable = true)
 |-- COUNTY: string (nullable = true)
 |-- ZIP: string (nullable = true)
 |-- LAT: string (nullable = true)
 |-- LON: string (nullable = true)
 |-- HEALTHCARE_EXPENSES: string (nullable = true)
 |-- HEALTHCARE_C

Id,BIRTHDATE,DEATHDATE,SSN,DRIVERS,PASSPORT,PREFIX,FIRST,LAST,SUFFIX,MAIDEN,MARITAL,RACE,ETHNICITY,GENDER,BIRTHPLACE,ADDRESS,CITY,STATE,COUNTY,ZIP,LAT,LON,HEALTHCARE_EXPENSES,HEALTHCARE_COVERAGE,_ingested_at,_source_file,_source_system
b9c610cd-28a6-4636-ccb6-c7a0d2a4cb85,2019-02-17,null,999-65-3251,null,null,null,Damon455,Langosh790,null,null,null,white,nonhispanic,M,Middleborough Massachusetts US,620 Lynch Tunnel Apt 0,Springfield,Massachusetts,Hampden County,01104,42.08038942501558,-72.48043144917739,9039.1645,7964.1255,2026-11-18T13:22:11.568Z,/Volumes/workspace/default/healthcare_source_data/patients.csv,Synthea
c1f1fcaa-82fd-d5b7-3544-c8f9708b06a8,2005-07-04,null,999-49-3323,S99941126,null,null,Thi53,Wunsch504,null,null,null,white,nonhispanic,F,Danvers Massachusetts US,972 Tillman Branch Suite 48,Bellingham,Massachusetts,Norfolk County,null,42.03521335752818,-71.48251904737748,402723.415,14064.135000000002,2026-11-18T13:22:11.568Z,/Volumes/workspace/default/healthcare_source_data/patients.csv,Synthea
339144f8-50e1-633e-a013-f361391c4cff,1998-05-11,null,999-10-8743,S99996708,X75063318X,Mr.,Chi716,Greenfelder433,null,null,null,white,nonhispanic,M,Athens Athens Prefecture GR,1060 Bernhard Crossroad Suite 15,Boston,Massachusetts,Suffolk County,02131,42.29255662362827,-71.06116042204106,571935.8725,787.5374999999999,2026-11-18T13:22:11.568Z,/Volumes/workspace/default/healthcare_source_data/patients.csv,Synthea


BRONZE TABLE: workspace.default.healthcare_bronze_encounters
root
 |-- Id: string (nullable = true)
 |-- START: string (nullable = true)
 |-- STOP: string (nullable = true)
 |-- PATIENT: string (nullable = true)
 |-- ORGANIZATION: string (nullable = true)
 |-- PROVIDER: string (nullable = true)
 |-- PAYER: string (nullable = true)
 |-- ENCOUNTERCLASS: string (nullable = true)
 |-- CODE: string (nullable = true)
 |-- DESCRIPTION: string (nullable = true)
 |-- BASE_ENCOUNTER_COST: string (nullable = true)
 |-- TOTAL_CLAIM_COST: string (nullable = true)
 |-- PAYER_COVERAGE: string (nullable = true)
 |-- REASONCODE: string (nullable = true)
 |-- REASONDESCRIPTION: string (nullable = true)
 |-- _ingested_at: timestamp (nullable = true)
 |-- _source_file: string (nullable = true)
 |-- _source_system: string (nullable = true)



Id,START,STOP,PATIENT,ORGANIZATION,PROVIDER,PAYER,ENCOUNTERCLASS,CODE,DESCRIPTION,BASE_ENCOUNTER_COST,TOTAL_CLAIM_COST,PAYER_COVERAGE,REASONCODE,REASONDESCRIPTION,_ingested_at,_source_file,_source_system
748f8357-6cc7-551d-f31a-32fa2cf84126,2019-02-17T05:07:38Z,2019-02-17T05:22:38Z,b9c610cd-28a6-4636-ccb6-c7a0d2a4cb85,f7ae497d-8dc6-3721-9402-43b621a4e7d2,82608ebb-037c-3cef-9d34-3736d69b29e8,7c4411ce-02f1-39b5-b9ec-dfbea9ad3c1a,wellness,410620009,Well child visit (procedure),129.16,877.79,833.90,null,null,2026-11-18T13:22:14.232Z,/Volumes/workspace/default/healthcare_source_data/encounters.csv,Synthea
5a4735ae-423f-6563-28ab-b3d11b49b2d4,2019-03-24T05:07:38Z,2019-03-24T05:22:38Z,b9c610cd-28a6-4636-ccb6-c7a0d2a4cb85,f7ae497d-8dc6-3721-9402-43b621a4e7d2,82608ebb-037c-3cef-9d34-3736d69b29e8,7c4411ce-02f1-39b5-b9ec-dfbea9ad3c1a,wellness,410620009,Well child visit (procedure),129.16,269.68,256.20,null,null,2026-11-18T13:22:14.232Z,/Volumes/workspace/default/healthcare_source_data/encounters.csv,Synthea
0bee1ce6-3e2c-5506-f71c-a7ba8f64a3d3,2019-05-26T05:07:38Z,2019-05-26T05:22:38Z,b9c610cd-28a6-4636-ccb6-c7a0d2a4cb85,f7ae497d-8dc6-3721-9402-43b621a4e7d2,82608ebb-037c-3cef-9d34-3736d69b29e8,7c4411ce-02f1-39b5-b9ec-dfbea9ad3c1a,wellness,410620009,Well child visit (procedure),129.16,1292.75,1228.11,null,null,2026-11-18T13:22:14.232Z,/Volumes/workspace/default/healthcare_source_data/encounters.csv,Synthea


BRONZE TABLE: workspace.default.healthcare_bronze_conditions
root
 |-- START: string (nullable = true)
 |-- STOP: string (nullable = true)
 |-- PATIENT: string (nullable = true)
 |-- ENCOUNTER: string (nullable = true)
 |-- CODE: string (nullable = true)
 |-- DESCRIPTION: string (nullable = true)
 |-- _ingested_at: timestamp (nullable = true)
 |-- _source_file: string (nullable = true)
 |-- _source_system: string (nullable = true)



START,STOP,PATIENT,ENCOUNTER,CODE,DESCRIPTION,_ingested_at,_source_file,_source_system
2013-06-24,2013-07-02,c1f1fcaa-82fd-d5b7-3544-c8f9708b06a8,0b2794bd-ec2b-d34f-0610-2523b3b7fcf0,10509002,Acute bronchitis (disorder),2026-11-18T13:22:17.208Z,/Volumes/workspace/default/healthcare_source_data/conditions.csv,Synthea
2016-02-27,2016-03-14,c1f1fcaa-82fd-d5b7-3544-c8f9708b06a8,a6d818dd-0983-fd1c-eefa-3d2295532c45,283371005,Laceration of forearm,2026-11-18T13:22:17.208Z,/Volumes/workspace/default/healthcare_source_data/conditions.csv,Synthea
2016-08-11,2016-08-22,c1f1fcaa-82fd-d5b7-3544-c8f9708b06a8,36d2e781-4655-0a11-1f70-c69856e02019,444814009,Viral sinusitis (disorder),2026-11-18T13:22:17.208Z,/Volumes/workspace/default/healthcare_source_data/conditions.csv,Synthea


BRONZE TABLE: workspace.default.healthcare_bronze_procedures
root
 |-- START: string (nullable = true)
 |-- STOP: string (nullable = true)
 |-- PATIENT: string (nullable = true)
 |-- ENCOUNTER: string (nullable = true)
 |-- CODE: string (nullable = true)
 |-- DESCRIPTION: string (nullable = true)
 |-- BASE_COST: string (nullable = true)
 |-- REASONCODE: string (nullable = true)
 |-- REASONDESCRIPTION: string (nullable = true)
 |-- _ingested_at: timestamp (nullable = true)
 |-- _source_file: string (nullable = true)
 |-- _source_system: string (nullable = true)



START,STOP,PATIENT,ENCOUNTER,CODE,DESCRIPTION,BASE_COST,REASONCODE,REASONDESCRIPTION,_ingested_at,_source_file,_source_system
2019-02-17T05:07:38Z,2019-02-17T05:22:38Z,b9c610cd-28a6-4636-ccb6-c7a0d2a4cb85,748f8357-6cc7-551d-f31a-32fa2cf84126,430193006,Medication Reconciliation (procedure),608.11,null,null,2026-11-18T13:22:19.610Z,/Volumes/workspace/default/healthcare_source_data/procedures.csv,Synthea
2019-05-26T05:07:38Z,2019-05-26T05:22:38Z,b9c610cd-28a6-4636-ccb6-c7a0d2a4cb85,0bee1ce6-3e2c-5506-f71c-a7ba8f64a3d3,430193006,Medication Reconciliation (procedure),460.99,null,null,2026-11-18T13:22:19.610Z,/Volumes/workspace/default/healthcare_source_data/procedures.csv,Synthea
2019-07-28T05:07:38Z,2019-07-28T05:22:38Z,b9c610cd-28a6-4636-ccb6-c7a0d2a4cb85,6e93bcf9-45a4-8528-0120-1c1eaa930faf,430193006,Medication Reconciliation (procedure),492.11,null,null,2026-11-18T13:22:19.610Z,/Volumes/workspace/default/healthcare_source_data/procedures.csv,Synthea


In [0]:
# Load existing Bronze Delta tables for Silver processing
#creaing the reference to our existing bronze created tables

bronze = {
    "patients": spark.table("workspace.default.healthcare_bronze_patients"),
    "encounters": spark.table("workspace.default.healthcare_bronze_encounters"),
    "conditions": spark.table("workspace.default.healthcare_bronze_conditions"),
    "procedures": spark.table("workspace.default.healthcare_bronze_procedures")
}

print("Bronze tables successfully loaded for Silver processing.")

Bronze tables successfully loaded for Silver processing.


In [0]:

from pyspark.sql import functions as F

# --------------------------------------------------
# STAGE 4.2 - SILVER PATIENTS TRANSFORMATION
# --------------------------------------------------

patients_silver = (
    bronze["patients"]

    # Select only required columns and standardize names
    .select(
        F.col("Id").alias("patient_id"),

        F.to_date(F.trim(F.col("BIRTHDATE"))).alias("birth_date"),
        F.to_date(F.trim(F.col("DEATHDATE"))).alias("death_date"),

        F.trim(F.col("FIRST")).alias("first_name"),
        F.trim(F.col("LAST")).alias("last_name"),

        F.trim(F.col("MARITAL")).alias("marital_status"),
        F.trim(F.col("RACE")).alias("race"),
        F.trim(F.col("ETHNICITY")).alias("ethnicity"),
        F.trim(F.col("GENDER")).alias("gender"),

        F.trim(F.col("BIRTHPLACE")).alias("birthplace"),

        F.trim(F.col("ADDRESS")).alias("address"),
        F.trim(F.col("CITY")).alias("city"),
        F.trim(F.col("STATE")).alias("state"),
        F.trim(F.col("COUNTY")).alias("county"),
        F.trim(F.col("ZIP")).alias("zip_code"),

        F.expr("try_cast(LAT as double)").alias("latitude"),
        F.expr("try_cast(LON as double)").alias("longitude"),

        F.expr("try_cast(HEALTHCARE_EXPENSES as decimal(18,2))")
         .alias("healthcare_expenses"),

        F.expr("try_cast(HEALTHCARE_COVERAGE as decimal(18,2))")
         .alias("healthcare_coverage"),

        # Preserve ingestion lineage
        F.col("_ingested_at"),
        F.col("_source_file"),
        F.col('_source_system')
    )
)

# Save the transformed dataset as a Delta table
(
    patients_silver.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable("workspace.default.healthcare_silver_patients")
)

print("Silver Patients table created successfully.")


Silver Patients table created successfully.


In [0]:
silver_patients = spark.table(
    "workspace.default.healthcare_silver_patients"
)

print("Silver Patients Schema:")
silver_patients.printSchema()

print("Total Patients:", silver_patients.count())

display(silver_patients.limit(10))

Silver Patients Schema:
root
 |-- patient_id: string (nullable = true)
 |-- birth_date: date (nullable = true)
 |-- death_date: date (nullable = true)
 |-- first_name: string (nullable = true)
 |-- last_name: string (nullable = true)
 |-- marital_status: string (nullable = true)
 |-- race: string (nullable = true)
 |-- ethnicity: string (nullable = true)
 |-- gender: string (nullable = true)
 |-- birthplace: string (nullable = true)
 |-- address: string (nullable = true)
 |-- city: string (nullable = true)
 |-- state: string (nullable = true)
 |-- county: string (nullable = true)
 |-- zip_code: string (nullable = true)
 |-- latitude: double (nullable = true)
 |-- longitude: double (nullable = true)
 |-- healthcare_expenses: decimal(18,2) (nullable = true)
 |-- healthcare_coverage: decimal(18,2) (nullable = true)
 |-- _ingested_at: timestamp (nullable = true)
 |-- _source_file: string (nullable = true)
 |-- _source_system: string (nullable = true)

Total Patients: 1163


patient_id,birth_date,death_date,first_name,last_name,marital_status,race,ethnicity,gender,birthplace,address,city,state,county,zip_code,latitude,longitude,healthcare_expenses,healthcare_coverage,_ingested_at,_source_file,_source_system
b9c610cd-28a6-4636-ccb6-c7a0d2a4cb85,2019-02-17,null,Damon455,Langosh790,null,white,nonhispanic,M,Middleborough Massachusetts US,620 Lynch Tunnel Apt 0,Springfield,Massachusetts,Hampden County,01104,42.08038942501558,-72.48043144917739,9039.16,7964.13,2026-11-18T13:22:11.568Z,/Volumes/workspace/default/healthcare_source_data/patients.csv,Synthea
c1f1fcaa-82fd-d5b7-3544-c8f9708b06a8,2005-07-04,null,Thi53,Wunsch504,null,white,nonhispanic,F,Danvers Massachusetts US,972 Tillman Branch Suite 48,Bellingham,Massachusetts,Norfolk County,null,42.03521335752818,-71.48251904737748,402723.42,14064.14,2026-11-18T13:22:11.568Z,/Volumes/workspace/default/healthcare_source_data/patients.csv,Synthea
339144f8-50e1-633e-a013-f361391c4cff,1998-05-11,null,Chi716,Greenfelder433,null,white,nonhispanic,M,Athens Athens Prefecture GR,1060 Bernhard Crossroad Suite 15,Boston,Massachusetts,Suffolk County,02131,42.29255662362827,-71.06116042204106,571935.87,787.54,2026-11-18T13:22:11.568Z,/Volumes/workspace/default/healthcare_source_data/patients.csv,Synthea
d488232e-bf14-4bed-08c0-a82f34b6a197,2003-01-28,null,Phillis443,Walter473,null,white,nonhispanic,F,Boston Massachusetts US,677 Ritchie Terrace,Hingham,Massachusetts,Plymouth County,02043,42.200490824321214,-70.91607557520302,582557.80,104782.21,2026-11-18T13:22:11.568Z,/Volumes/workspace/default/healthcare_source_data/patients.csv,Synthea
217f95a3-4e10-bd5d-fb67-0cfb5e8ba075,1993-12-23,null,Jerrold404,Herzog843,M,black,nonhispanic,M,Boston Massachusetts US,276 Bernier Branch,Revere,Massachusetts,Suffolk County,null,42.3818748756275,-70.99928578468784,475826.86,18067.09,2026-11-18T13:22:11.568Z,/Volumes/workspace/default/healthcare_source_data/patients.csv,Synthea
faac724a-a9e9-be66-fe1e-3044dc0ba8ea,2020-12-04,null,Brandon214,Watsica258,null,white,nonhispanic,F,Northborough Massachusetts US,734 Brakus Pathway Unit 28,New Marlborough,Massachusetts,Berkshire County,null,42.147028011364384,-73.21710320353982,31468.04,206.22,2026-11-18T13:22:11.568Z,/Volumes/workspace/default/healthcare_source_data/patients.csv,Synthea
23d16ee3-8cd4-eeb8-e77e-1e5fbf4c4159,1990-12-15,null,Rodrigo242,Arellano2,M,black,hispanic,M,Bogota Bogota CO,538 Crona Underpass,Revere,Massachusetts,Suffolk County,02151,42.44326150073713,-71.02232158016432,772362.73,2213.09,2026-11-18T13:22:11.568Z,/Volumes/workspace/default/healthcare_source_data/patients.csv,Synthea
aade3c61-92bd-d079-9d28-0b2b7fde0fbb,1985-06-05,null,Sal878,Hoppe518,M,white,nonhispanic,M,Merrimac Massachusetts US,930 Goldner Stravenue Apt 8,Somerville,Massachusetts,Middlesex County,02138,42.363459871015344,-71.1022528716212,924957.27,874.37,2026-11-18T13:22:11.568Z,/Volumes/workspace/default/healthcare_source_data/patients.csv,Synthea
0288c42c-43a1-9878-4a9d-6b96caa12c40,1979-12-17,null,Miki234,Kozey370,M,white,nonhispanic,F,Northbridge Massachusetts US,636 Stamm Gateway,Cambridge,Massachusetts,Middlesex County,02472,42.337425423321065,-71.1066547010483,1134817.57,226909.96,2026-11-18T13:22:11.568Z,/Volumes/workspace/default/healthcare_source_data/patients.csv,Synthea
dc6c06d0-a7d8-100f-c08b-46c93700c188,2006-07-11,null,Taylor21,Gulgowski816,null,white,nonhispanic,M,East Longmeadow Massachusetts US,1037 Hills Extension,Springfield,Massachusetts,Hampden County,01013,42.13505594447831,-72.48174986407135,390675.58,1826.33,2026-11-18T13:22:11.568Z,/Volumes/workspace/default/healthcare_source_data/patients.csv,Synthea


In [0]:

from pyspark.sql import functions as F

# ------------------------------------------------------------
# STAGE 4.2.1 - SILVER PATIENTS TRANSFORMATION VALIDATION
# Focus: Transformation integrity only (no repeated source DQ)
# ------------------------------------------------------------

bronze_patients = bronze["patients"]
silver_patients = spark.table(
    "workspace.default.healthcare_silver_patients"
)

# 1. Bronze vs Silver record count
bronze_count = bronze_patients.count()
silver_count = silver_patients.count()
record_count_status = bronze_count == silver_count

# 2. Date conversion failures
date_conversion_failures = (
    bronze_patients.alias("b")
    .join(
        silver_patients.select("patient_id", "birth_date", "death_date").alias("s"),
        F.col("b.Id") == F.col("s.patient_id"),
        "left"
    )
    .filter(
        (
            F.col("b.BIRTHDATE").isNotNull() &
            F.col("s.birth_date").isNull()
        ) |
        (
            F.col("b.DEATHDATE").isNotNull() &
            F.col("s.death_date").isNull()
        )
    )
    .count()
)

# 3. Numeric conversion failures
numeric_conversion_failures = (
    bronze_patients.alias("b")
    .join(
        silver_patients.select(
            "patient_id", "latitude", "longitude",
            "healthcare_expenses", "healthcare_coverage"
        ).alias("s"),
        F.col("b.Id") == F.col("s.patient_id"),
        "left"
    )
    .filter(
        (
            F.col("b.LAT").isNotNull() &
            F.col("s.latitude").isNull()
        ) |
        (
            F.col("b.LON").isNotNull() &
            F.col("s.longitude").isNull()
        ) |
        (
            F.col("b.HEALTHCARE_EXPENSES").isNotNull() &
            F.col("s.healthcare_expenses").isNull()
        ) |
        (
            F.col("b.HEALTHCARE_COVERAGE").isNotNull() &
            F.col("s.healthcare_coverage").isNull()
        )
    )
    .count()
)

# 4. Schema verification
expected_types = {
    "patient_id": "string",
    "birth_date": "date",
    "death_date": "date",
    "latitude": "double",
    "longitude": "double",
    "healthcare_expenses": "decimal(18,2)",
    "healthcare_coverage": "decimal(18,2)"
}

actual_types = {
    field.name: field.dataType.simpleString()
    for field in silver_patients.schema.fields
}

schema_failures = [
    f"{column}: expected {expected}, found {actual_types.get(column)}"
    for column, expected in expected_types.items()
    if actual_types.get(column) != expected
]

schema_status = len(schema_failures) == 0

# 5. Metadata preservation
metadata_columns = [
    "_ingested_at",
    "_source_file",
    "_source_system"
]

missing_metadata = [
    column for column in metadata_columns
    if column not in silver_patients.columns
]

metadata_status = len(missing_metadata) == 0

# 6. Sensitive-column exclusion
sensitive_columns = [
    "SSN", "DRIVERS", "PASSPORT",
    "PREFIX", "SUFFIX", "MAIDEN"
]

excluded_sensitive_columns = [
    column for column in sensitive_columns
    if column in silver_patients.columns
]

sensitive_status = len(excluded_sensitive_columns) == 0

# ------------------------------------------------------------
# FINAL VALIDATION REPORT
# ------------------------------------------------------------


validation_results = [
    ("Bronze vs Silver Record Count",
     f"Bronze={bronze_count}, Silver={silver_count}",
     "PASS" if record_count_status else "FAIL"),

    ("Date Conversion Failures",
     str(date_conversion_failures),
     "PASS" if date_conversion_failures == 0 else "REVIEW"),

    ("Numeric Conversion Failures",
     str(numeric_conversion_failures),
     "PASS" if numeric_conversion_failures == 0 else "REVIEW"),

    ("Schema Verification",
     str(schema_failures) if schema_failures else "All expected data types match",
     "PASS" if schema_status else "FAIL"),

    ("Metadata Preservation",
     str(missing_metadata) if missing_metadata else "All lineage columns present",
     "PASS" if metadata_status else "FAIL"),

    ("Sensitive Column Exclusion",
     str(excluded_sensitive_columns) if excluded_sensitive_columns else "Sensitive columns excluded",
     "PASS" if sensitive_status else "FAIL")
]

validation_report = spark.createDataFrame(
    validation_results,
    ["Validation_Check", "Result", "Status"]
)

display(validation_report)



Validation_Check,Result,Status
Bronze vs Silver Record Count,"Bronze=1163, Silver=1163",PASS
Date Conversion Failures,0,PASS
Numeric Conversion Failures,0,PASS
Schema Verification,All expected data types match,PASS
Metadata Preservation,All lineage columns present,PASS
Sensitive Column Exclusion,Sensitive columns excluded,PASS


In [0]:

from pyspark.sql import functions as F

# --------------------------------------------------
# STAGE 4.3 - SILVER ENCOUNTERS TRANSFORMATION
# --------------------------------------------------

encounters_silver = (
    bronze["encounters"]

    .select(
        F.col("Id").alias("encounter_id"),

        F.expr("try_cast(START as timestamp)")
         .alias("start_datetime"),

        F.expr("try_cast(STOP as timestamp)")
         .alias("stop_datetime"),

        F.col("PATIENT").alias("patient_id"),
        F.col("ORGANIZATION").alias("organization_id"),
        F.col("PROVIDER").alias("provider_id"),
        F.col("PAYER").alias("payer_id"),

        F.trim(F.col("ENCOUNTERCLASS"))
         .alias("encounter_class"),

        F.col("CODE").alias("encounter_code"),
        F.trim(F.col("DESCRIPTION"))
         .alias("encounter_description"),

        F.expr("try_cast(BASE_ENCOUNTER_COST as decimal(18,2))")
         .alias("base_encounter_cost"),

        F.expr("try_cast(TOTAL_CLAIM_COST as decimal(18,2))")
         .alias("total_claim_cost"),

        F.expr("try_cast(PAYER_COVERAGE as decimal(18,2))")
         .alias("payer_coverage"),

        F.col("REASONCODE").alias("reason_code"),
        F.trim(F.col("REASONDESCRIPTION"))
         .alias("reason_description"),

        # Preserve data lineage
        F.col("_ingested_at"),
        F.col("_source_file"),
        F.col("_source_system")
    )
)

# Persist as Silver Delta table
(
    encounters_silver.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable("workspace.default.healthcare_silver_encounters")
)

print("Silver Encounters table created successfully.")


Silver Encounters table created successfully.


In [0]:

# --------------------------------------------------
# STAGE 4.3.1 - SILVER ENCOUNTERS VERIFICATION
# --------------------------------------------------

silver_encounters = spark.table(
    "workspace.default.healthcare_silver_encounters"
)

print("Silver Encounters Schema:")
silver_encounters.printSchema()

print("Total Encounters:", silver_encounters.count())

display(silver_encounters.limit(10))


Silver Encounters Schema:
root
 |-- encounter_id: string (nullable = true)
 |-- start_datetime: timestamp (nullable = true)
 |-- stop_datetime: timestamp (nullable = true)
 |-- patient_id: string (nullable = true)
 |-- organization_id: string (nullable = true)
 |-- provider_id: string (nullable = true)
 |-- payer_id: string (nullable = true)
 |-- encounter_class: string (nullable = true)
 |-- encounter_code: string (nullable = true)
 |-- encounter_description: string (nullable = true)
 |-- base_encounter_cost: decimal(18,2) (nullable = true)
 |-- total_claim_cost: decimal(18,2) (nullable = true)
 |-- payer_coverage: decimal(18,2) (nullable = true)
 |-- reason_code: string (nullable = true)
 |-- reason_description: string (nullable = true)
 |-- _ingested_at: timestamp (nullable = true)
 |-- _source_file: string (nullable = true)
 |-- _source_system: string (nullable = true)

Total Encounters: 61459


encounter_id,start_datetime,stop_datetime,patient_id,organization_id,provider_id,payer_id,encounter_class,encounter_code,encounter_description,base_encounter_cost,total_claim_cost,payer_coverage,reason_code,reason_description,_ingested_at,_source_file,_source_system
748f8357-6cc7-551d-f31a-32fa2cf84126,2019-02-17T05:07:38.000Z,2019-02-17T05:22:38.000Z,b9c610cd-28a6-4636-ccb6-c7a0d2a4cb85,f7ae497d-8dc6-3721-9402-43b621a4e7d2,82608ebb-037c-3cef-9d34-3736d69b29e8,7c4411ce-02f1-39b5-b9ec-dfbea9ad3c1a,wellness,410620009,Well child visit (procedure),129.16,877.79,833.90,null,null,2026-11-18T13:22:14.232Z,/Volumes/workspace/default/healthcare_source_data/encounters.csv,Synthea
5a4735ae-423f-6563-28ab-b3d11b49b2d4,2019-03-24T05:07:38.000Z,2019-03-24T05:22:38.000Z,b9c610cd-28a6-4636-ccb6-c7a0d2a4cb85,f7ae497d-8dc6-3721-9402-43b621a4e7d2,82608ebb-037c-3cef-9d34-3736d69b29e8,7c4411ce-02f1-39b5-b9ec-dfbea9ad3c1a,wellness,410620009,Well child visit (procedure),129.16,269.68,256.20,null,null,2026-11-18T13:22:14.232Z,/Volumes/workspace/default/healthcare_source_data/encounters.csv,Synthea
0bee1ce6-3e2c-5506-f71c-a7ba8f64a3d3,2019-05-26T05:07:38.000Z,2019-05-26T05:22:38.000Z,b9c610cd-28a6-4636-ccb6-c7a0d2a4cb85,f7ae497d-8dc6-3721-9402-43b621a4e7d2,82608ebb-037c-3cef-9d34-3736d69b29e8,7c4411ce-02f1-39b5-b9ec-dfbea9ad3c1a,wellness,410620009,Well child visit (procedure),129.16,1292.75,1228.11,null,null,2026-11-18T13:22:14.232Z,/Volumes/workspace/default/healthcare_source_data/encounters.csv,Synthea
6e93bcf9-45a4-8528-0120-1c1eaa930faf,2019-07-28T05:07:38.000Z,2019-07-28T05:22:38.000Z,b9c610cd-28a6-4636-ccb6-c7a0d2a4cb85,f7ae497d-8dc6-3721-9402-43b621a4e7d2,82608ebb-037c-3cef-9d34-3736d69b29e8,7c4411ce-02f1-39b5-b9ec-dfbea9ad3c1a,wellness,410620009,Well child visit (procedure),129.16,1323.87,1257.68,null,null,2026-11-18T13:22:14.232Z,/Volumes/workspace/default/healthcare_source_data/encounters.csv,Synthea
8b6787c3-4316-a0cb-899d-4746525c319f,2019-10-27T05:07:38.000Z,2019-10-27T05:22:38.000Z,b9c610cd-28a6-4636-ccb6-c7a0d2a4cb85,f7ae497d-8dc6-3721-9402-43b621a4e7d2,82608ebb-037c-3cef-9d34-3736d69b29e8,7c4411ce-02f1-39b5-b9ec-dfbea9ad3c1a,wellness,410620009,Well child visit (procedure),129.16,831.76,790.17,null,null,2026-11-18T13:22:14.232Z,/Volumes/workspace/default/healthcare_source_data/encounters.csv,Synthea
8f424287-ee3a-c144-bc1d-3ba926e93fd5,2020-01-26T05:07:38.000Z,2020-01-26T05:22:38.000Z,b9c610cd-28a6-4636-ccb6-c7a0d2a4cb85,f7ae497d-8dc6-3721-9402-43b621a4e7d2,82608ebb-037c-3cef-9d34-3736d69b29e8,7c4411ce-02f1-39b5-b9ec-dfbea9ad3c1a,wellness,410620009,Well child visit (procedure),129.16,129.16,122.70,null,null,2026-11-18T13:22:14.232Z,/Volumes/workspace/default/healthcare_source_data/encounters.csv,Synthea
fb15e123-fea7-cae8-6d49-ee9d2a85fc84,2020-02-05T06:07:38.000Z,2020-02-05T06:22:38.000Z,b9c610cd-28a6-4636-ccb6-c7a0d2a4cb85,5103c940-0c08-392f-95cd-446e0cea042a,9f529022-166f-3db4-9f92-ff1ba11ee565,7c4411ce-02f1-39b5-b9ec-dfbea9ad3c1a,ambulatory,185347001,Encounter for problem,77.49,77.49,16.62,null,null,2026-11-18T13:22:14.232Z,/Volumes/workspace/default/healthcare_source_data/encounters.csv,Synthea
01efcc52-15d6-51e9-faa2-bee069fcbe44,2020-02-17T10:07:38.000Z,2020-02-17T10:40:32.000Z,b9c610cd-28a6-4636-ccb6-c7a0d2a4cb85,5103c940-0c08-392f-95cd-446e0cea042a,9f529022-166f-3db4-9f92-ff1ba11ee565,7c4411ce-02f1-39b5-b9ec-dfbea9ad3c1a,ambulatory,185347001,Encounter for problem,77.49,594.14,507.43,null,null,2026-11-18T13:22:14.232Z,/Volumes/workspace/default/healthcare_source_data/encounters.csv,Synthea
1a7debfc-9582-7f23-a109-4f154a182ee2,2020-04-26T05:07:38.000Z,2020-04-26T05:22:38.000Z,b9c610cd-28a6-4636-ccb6-c7a0d2a4cb85,f7ae497d-8dc6-3721-9402-43b621a4e7d2,82608ebb-037c-3cef-9d34-3736d69b29e8,7c4411ce-02f1-39b5-b9ec-dfbea9ad3c1a,wellness,410620009,Well child visit (procedure),129.16,1614.96,1534.21,null,null,2026-11-18T13:22:14.232Z,/Volumes/workspace/default/healthcare_source_data/encounters.csv,Synthea
bf38c711-941f-7509-f9ec-b864d6929

In [0]:

from pyspark.sql import functions as F

# --------------------------------------------------
# STAGE 4.3.1 - SILVER ENCOUNTERS VALIDATION
# --------------------------------------------------

bronze_encounters = bronze["encounters"]
silver_encounters = spark.table(
    "workspace.default.healthcare_silver_encounters"
)

# 1. Record count
bronze_count = bronze_encounters.count()
silver_count = silver_encounters.count()

record_count_pass = bronze_count == silver_count

# 2. Timestamp conversion failures
date_conversion_failures = (
    bronze_encounters
    .filter(
        (F.col("START").isNotNull() &
         F.expr("try_cast(START as timestamp)").isNull())
        |
        (F.col("STOP").isNotNull() &
         F.expr("try_cast(STOP as timestamp)").isNull())
    )
    .count()
)

# 3. Financial conversion failures
numeric_conversion_failures = (
    bronze_encounters
    .filter(
        (F.col("BASE_ENCOUNTER_COST").isNotNull() &
         F.expr("try_cast(BASE_ENCOUNTER_COST as decimal(18,2))").isNull())
        |
        (F.col("TOTAL_CLAIM_COST").isNotNull() &
         F.expr("try_cast(TOTAL_CLAIM_COST as decimal(18,2))").isNull())
        |
        (F.col("PAYER_COVERAGE").isNotNull() &
         F.expr("try_cast(PAYER_COVERAGE as decimal(18,2))").isNull())
    )
    .count()
)

# 4. Invalid encounter duration
invalid_duration_count = (
    silver_encounters
    .filter(
        F.col("stop_datetime").isNotNull() &
        F.col("start_datetime").isNotNull() &
        (F.col("stop_datetime") < F.col("start_datetime"))
    )
    .count()
)

# 5. Patient referential integrity
silver_patients = spark.table(
    "workspace.default.healthcare_silver_patients"
)

orphan_patient_count = (
    silver_encounters
    .select("patient_id")
    .distinct()
    .join(
        silver_patients.select("patient_id").distinct(),
        on="patient_id",
        how="left_anti"
    )
    .count()
)

# 6. Schema verification
expected_types = {
    "encounter_id": "string",
    "start_datetime": "timestamp",
    "stop_datetime": "timestamp",
    "patient_id": "string",
    "base_encounter_cost": "decimal(18,2)",
    "total_claim_cost": "decimal(18,2)",
    "payer_coverage": "decimal(18,2)",
    "_ingested_at": "timestamp",
    "_source_file": "string",
    "_source_system": "string"
}

actual_types = {
    field.name: field.dataType.simpleString()
    for field in silver_encounters.schema.fields
}

schema_failures = {
    column: (expected, actual_types.get(column))
    for column, expected in expected_types.items()
    if actual_types.get(column) != expected
}

schema_pass = len(schema_failures) == 0

# 7. Metadata preservation
metadata_columns = [
    "_ingested_at",
    "_source_file",
    "_source_system"
]

missing_metadata = [
    col for col in metadata_columns
    if col not in silver_encounters.columns
]

metadata_pass = len(missing_metadata) == 0

# --------------------------------------------------
# VALIDATION REPORT
# --------------------------------------------------

validation_results = [
    (
        "Bronze vs Silver Record Count",
        f"Bronze={bronze_count}, Silver={silver_count}",
        "PASS" if record_count_pass else "FAIL"
    ),
    (
        "Timestamp Conversion Failures",
        str(date_conversion_failures),
        "PASS" if date_conversion_failures == 0 else "FAIL"
    ),
    (
        "Numeric Conversion Failures",
        str(numeric_conversion_failures),
        "PASS" if numeric_conversion_failures == 0 else "FAIL"
    ),
    (
        "Invalid Encounter Durations",
        str(invalid_duration_count),
        "PASS" if invalid_duration_count == 0 else "REVIEW"
    ),
    (
        "Orphan Patient References",
        str(orphan_patient_count),
        "PASS" if orphan_patient_count == 0 else "REVIEW"
    ),
    (
        "Schema Verification",
        "Schema matches expected types"
        if schema_pass else str(schema_failures),
        "PASS" if schema_pass else "FAIL"
    ),
    (
        "Metadata Preservation",
        "All lineage columns present"
        if metadata_pass else str(missing_metadata),
        "PASS" if metadata_pass else "FAIL"
    )
]

encounter_validation_report = spark.createDataFrame(
    validation_results,
    ["Validation_Check", "Result", "Status"]
)

display(encounter_validation_report)


Validation_Check,Result,Status
Bronze vs Silver Record Count,"Bronze=61459, Silver=61459",PASS
Timestamp Conversion Failures,0,PASS
Numeric Conversion Failures,0,PASS
Invalid Encounter Durations,0,PASS
Orphan Patient References,0,PASS
Schema Verification,Schema matches expected types,PASS
Metadata Preservation,All lineage columns present,PASS


In [0]:

from pyspark.sql import functions as F

# --------------------------------------------------
# STAGE 4.4 - SILVER CONDITIONS TRANSFORMATION
# --------------------------------------------------

conditions_silver = (
    bronze["conditions"]
    .select(
        F.expr("try_cast(START as date)")
         .alias("condition_start_date"),

        F.expr("try_cast(STOP as date)")
         .alias("condition_end_date"),

        F.col("PATIENT").alias("patient_id"),
        F.col("ENCOUNTER").alias("encounter_id"),
        F.col("CODE").alias("condition_code"),
        F.trim(F.col("DESCRIPTION"))
         .alias("condition_description"),

        # Preserve source lineage
        F.col("_ingested_at"),
        F.col("_source_file"),
        F.col("_source_system")
    )
)

# Write Silver Delta table
(
    conditions_silver.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable("workspace.default.healthcare_silver_conditions")
)

print("Silver Conditions table created successfully.")


Silver Conditions table created successfully.


In [0]:

# --------------------------------------------------
# STAGE 4.4.1 - SILVER CONDITIONS VERIFICATION
# --------------------------------------------------

silver_conditions = spark.table(
    "workspace.default.healthcare_silver_conditions"
)

print("Silver Conditions Schema:")
silver_conditions.printSchema()

print("Total Conditions:", silver_conditions.count())

display(silver_conditions.limit(10))


Silver Conditions Schema:
root
 |-- condition_start_date: date (nullable = true)
 |-- condition_end_date: date (nullable = true)
 |-- patient_id: string (nullable = true)
 |-- encounter_id: string (nullable = true)
 |-- condition_code: string (nullable = true)
 |-- condition_description: string (nullable = true)
 |-- _ingested_at: timestamp (nullable = true)
 |-- _source_file: string (nullable = true)
 |-- _source_system: string (nullable = true)

Total Conditions: 38094


condition_start_date,condition_end_date,patient_id,encounter_id,condition_code,condition_description,_ingested_at,_source_file,_source_system
2013-06-24,2013-07-02,c1f1fcaa-82fd-d5b7-3544-c8f9708b06a8,0b2794bd-ec2b-d34f-0610-2523b3b7fcf0,10509002,Acute bronchitis (disorder),2026-11-18T13:22:17.208Z,/Volumes/workspace/default/healthcare_source_data/conditions.csv,Synthea
2016-02-27,2016-03-14,c1f1fcaa-82fd-d5b7-3544-c8f9708b06a8,a6d818dd-0983-fd1c-eefa-3d2295532c45,283371005,Laceration of forearm,2026-11-18T13:22:17.208Z,/Volumes/workspace/default/healthcare_source_data/conditions.csv,Synthea
2016-08-11,2016-08-22,c1f1fcaa-82fd-d5b7-3544-c8f9708b06a8,36d2e781-4655-0a11-1f70-c69856e02019,444814009,Viral sinusitis (disorder),2026-11-18T13:22:17.208Z,/Volumes/workspace/default/healthcare_source_data/conditions.csv,Synthea
2016-11-27,2016-12-17,c1f1fcaa-82fd-d5b7-3544-c8f9708b06a8,c8eaaf41-958b-31ab-7de5-568cee8751f3,444814009,Viral sinusitis (disorder),2026-11-18T13:22:17.208Z,/Volumes/workspace/default/healthcare_source_data/conditions.csv,Synthea
2017-02-22,2017-06-02,c1f1fcaa-82fd-d5b7-3544-c8f9708b06a8,6474f606-5a1b-48c0-bbbf-ad6dcbc24d4e,16114001,Fracture of ankle,2026-11-18T13:22:17.208Z,/Volumes/workspace/default/healthcare_source_data/conditions.csv,Synthea
2020-02-06,2020-03-14,c1f1fcaa-82fd-d5b7-3544-c8f9708b06a8,a6da4c61-bc91-17a7-14a2-fda9023536a3,44465007,Sprain of ankle,2026-11-18T13:22:17.208Z,/Volumes/workspace/default/healthcare_source_data/conditions.csv,Synthea
2012-06-11,2013-06-17,339144f8-50e1-633e-a013-f361391c4cff,33782d5d-ba30-c2e5-c432-e36a8f6c1cc8,160968000,Risk activity involvement (finding),2026-11-18T13:22:17.208Z,/Volumes/workspace/default/healthcare_source_data/conditions.csv,Synthea
2012-08-07,2012-09-02,339144f8-50e1-633e-a013-f361391c4cff,4224e87e-d63e-4b30-a01b-03daffac9d26,44465007,Sprain of ankle,2026-11-18T13:22:17.208Z,/Volumes/workspace/default/healthcare_source_data/conditions.csv,Synthea
2015-06-29,null,339144f8-50e1-633e-a013-f361391c4cff,28f73bd9-73d3-de33-35a6-cabeb9a5795c,162864005,Body mass index 30+ - obesity (finding),2026-11-18T13:22:17.208Z,/Volumes/workspace/default/healthcare_source_data/conditions.csv,Synthea
2016-07-04,null,339144f8-50e1-633e-a013-f361391c4cff,8195411f-4a65-9245-07d3-1999e1c223df,59621000,Hypertension,2026-11-18T13:22:17.208Z,/Volumes/workspace/default/healthcare_source_data/conditions.csv,Synthea


In [0]:

from pyspark.sql import functions as F

# --------------------------------------------------
# STAGE 4.4.1 - SILVER CONDITIONS VALIDATION
# --------------------------------------------------

bronze_conditions = bronze["conditions"]

silver_conditions = spark.table(
    "workspace.default.healthcare_silver_conditions"
)

# 1. Record count
bronze_count = bronze_conditions.count()
silver_count = silver_conditions.count()

record_count_pass = bronze_count == silver_count

# 2. Date conversion failures
date_conversion_failures = (
    bronze_conditions
    .filter(
        (F.col("START").isNotNull() &
         F.expr("try_cast(START as date)").isNull())
        |
        (F.col("STOP").isNotNull() &
         F.expr("try_cast(STOP as date)").isNull())
    )
    .count()
)

# 3. Orphan patient references
silver_patients = spark.table(
    "workspace.default.healthcare_silver_patients"
)

orphan_patient_count = (
    silver_conditions
    .select("patient_id")
    .distinct()
    .join(
        silver_patients.select("patient_id").distinct(),
        on="patient_id",
        how="left_anti"
    )
    .count()
)

# 4. Orphan encounter references
silver_encounters = spark.table(
    "workspace.default.healthcare_silver_encounters"
)

orphan_encounter_count = (
    silver_conditions
    .select("encounter_id")
    .distinct()
    .join(
        silver_encounters.select("encounter_id").distinct(),
        on="encounter_id",
        how="left_anti"
    )
    .count()
)

# 5. Schema verification
expected_types = {
    "condition_start_date": "date",
    "condition_end_date": "date",
    "patient_id": "string",
    "encounter_id": "string",
    "condition_code": "string",
    "condition_description": "string",
    "_ingested_at": "timestamp",
    "_source_file": "string",
    "_source_system": "string"
}

actual_types = {
    field.name: field.dataType.simpleString()
    for field in silver_conditions.schema.fields
}

schema_failures = {
    column: (expected, actual_types.get(column))
    for column, expected in expected_types.items()
    if actual_types.get(column) != expected
}

schema_pass = len(schema_failures) == 0

# 6. Metadata preservation
metadata_columns = [
    "_ingested_at",
    "_source_file",
    "_source_system"
]

missing_metadata = [
    col for col in metadata_columns
    if col not in silver_conditions.columns
]

metadata_pass = len(missing_metadata) == 0

# --------------------------------------------------
# VALIDATION REPORT
# --------------------------------------------------

validation_results = [
    (
        "Bronze vs Silver Record Count",
        f"Bronze={bronze_count}, Silver={silver_count}",
        "PASS" if record_count_pass else "FAIL"
    ),
    (
        "Date Conversion Failures",
        str(date_conversion_failures),
        "PASS" if date_conversion_failures == 0 else "FAIL"
    ),
    (
        "Orphan Patient References",
        str(orphan_patient_count),
        "PASS" if orphan_patient_count == 0 else "REVIEW"
    ),
    (
        "Orphan Encounter References",
        str(orphan_encounter_count),
        "PASS" if orphan_encounter_count == 0 else "REVIEW"
    ),
    (
        "Schema Verification",
        "Schema matches expected types"
        if schema_pass else str(schema_failures),
        "PASS" if schema_pass else "FAIL"
    ),
    (
        "Metadata Preservation",
        "All lineage columns present"
        if metadata_pass else str(missing_metadata),
        "PASS" if metadata_pass else "FAIL"
    )
]

conditions_validation_report = spark.createDataFrame(
    validation_results,
    ["Validation_Check", "Result", "Status"]
)

display(conditions_validation_report)


Validation_Check,Result,Status
Bronze vs Silver Record Count,"Bronze=38094, Silver=38094",PASS
Date Conversion Failures,0,PASS
Orphan Patient References,0,PASS
Orphan Encounter References,0,PASS
Schema Verification,Schema matches expected types,PASS
Metadata Preservation,All lineage columns present,PASS


In [0]:
from pyspark.sql import functions as F

procedures_bronze = bronze["procedures"]

procedures_silver = procedures_bronze.select(
    F.to_timestamp(F.col("START")).alias("procedure_start_datetime"),
    F.to_timestamp(F.col("STOP")).alias("procedure_end_datetime"),
    F.col("PATIENT").alias("patient_id"),
    F.col("ENCOUNTER").alias("encounter_id"),
    F.col("CODE").alias("procedure_code"),
    F.col("DESCRIPTION").alias("procedure_description"),
    F.col("BASE_COST").cast("decimal(12,2)").alias("base_cost"),
    F.col("REASONCODE").alias("reason_code"),
    F.col("REASONDESCRIPTION").alias("reason_description"),
    F.col("_ingested_at"),
    F.col("_source_file"),
    F.col("_source_system")
)

procedures_silver.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("workspace.default.healthcare_silver_procedures")

print("Silver Procedures recreated successfully")

Silver Procedures recreated successfully


In [0]:
silver_procedures = spark.table(
    "workspace.default.healthcare_silver_procedures"
)

print("Silver Procedures Schema:")
silver_procedures.printSchema()

print(f"Total Procedures: {silver_procedures.count()}")

display(silver_procedures.limit(10))

Silver Procedures Schema:
root
 |-- procedure_start_datetime: timestamp (nullable = true)
 |-- procedure_end_datetime: timestamp (nullable = true)
 |-- patient_id: string (nullable = true)
 |-- encounter_id: string (nullable = true)
 |-- procedure_code: string (nullable = true)
 |-- procedure_description: string (nullable = true)
 |-- base_cost: decimal(12,2) (nullable = true)
 |-- reason_code: string (nullable = true)
 |-- reason_description: string (nullable = true)
 |-- _ingested_at: timestamp (nullable = true)
 |-- _source_file: string (nullable = true)
 |-- _source_system: string (nullable = true)

Total Procedures: 83823


procedure_start_datetime,procedure_end_datetime,patient_id,encounter_id,procedure_code,procedure_description,base_cost,reason_code,reason_description,_ingested_at,_source_file,_source_system
2019-02-17T05:07:38.000Z,2019-02-17T05:22:38.000Z,b9c610cd-28a6-4636-ccb6-c7a0d2a4cb85,748f8357-6cc7-551d-f31a-32fa2cf84126,430193006,Medication Reconciliation (procedure),608.11,null,null,2026-11-18T13:22:19.610Z,/Volumes/workspace/default/healthcare_source_data/procedures.csv,Synthea
2019-05-26T05:07:38.000Z,2019-05-26T05:22:38.000Z,b9c610cd-28a6-4636-ccb6-c7a0d2a4cb85,0bee1ce6-3e2c-5506-f71c-a7ba8f64a3d3,430193006,Medication Reconciliation (procedure),460.99,null,null,2026-11-18T13:22:19.610Z,/Volumes/workspace/default/healthcare_source_data/procedures.csv,Synthea
2019-07-28T05:07:38.000Z,2019-07-28T05:22:38.000Z,b9c610cd-28a6-4636-ccb6-c7a0d2a4cb85,6e93bcf9-45a4-8528-0120-1c1eaa930faf,430193006,Medication Reconciliation (procedure),492.11,null,null,2026-11-18T13:22:19.610Z,/Volumes/workspace/default/healthcare_source_data/procedures.csv,Synthea
2020-02-17T10:07:38.000Z,2020-02-17T10:40:32.000Z,b9c610cd-28a6-4636-ccb6-c7a0d2a4cb85,01efcc52-15d6-51e9-faa2-bee069fcbe44,395142003,Allergy screening test,516.65,null,null,2026-11-18T13:22:19.610Z,/Volumes/workspace/default/healthcare_source_data/procedures.csv,Synthea
2020-04-26T05:07:38.000Z,2020-04-26T05:22:38.000Z,b9c610cd-28a6-4636-ccb6-c7a0d2a4cb85,1a7debfc-9582-7f23-a109-4f154a182ee2,430193006,Medication Reconciliation (procedure),783.20,null,null,2026-11-18T13:22:19.610Z,/Volumes/workspace/default/healthcare_source_data/procedures.csv,Synthea
2021-01-24T05:07:38.000Z,2021-01-24T05:22:38.000Z,b9c610cd-28a6-4636-ccb6-c7a0d2a4cb85,7bb78da2-31b8-497d-bc08-25eca8a904f1,430193006,Medication Reconciliation (procedure),682.65,null,null,2026-11-18T13:22:19.610Z,/Volumes/workspace/default/healthcare_source_data/procedures.csv,Synthea
2013-06-24T06:20:52.000Z,2013-06-24T06:39:19.000Z,c1f1fcaa-82fd-d5b7-3544-c8f9708b06a8,0b2794bd-ec2b-d34f-0610-2523b3b7fcf0,23426006,Measurement of respiratory function (procedure),310.42,10509002,Acute bronchitis (disorder),2026-11-18T13:22:19.610Z,/Volumes/workspace/default/healthcare_source_data/procedures.csv,Synthea
2016-02-27T08:20:52.000Z,2016-02-27T08:38:46.000Z,c1f1fcaa-82fd-d5b7-3544-c8f9708b06a8,a6d818dd-0983-fd1c-eefa-3d2295532c45,288086009,Suture open wound,7788.10,283371005,Laceration of forearm,2026-11-18T13:22:19.610Z,/Volumes/workspace/default/healthcare_source_data/procedures.csv,Synthea
2016-02-27T08:38:46.000Z,2016-02-27T08:52:29.000Z,c1f1fcaa-82fd-d5b7-3544-c8f9708b06a8,a6d818dd-0983-fd1c-eefa-3d2295532c45,384700001,Injection of tetanus antitoxin,2750.11,283371005,Laceration of forearm,2026-11-18T13:22:19.610Z,/Volumes/workspace/default/healthcare_source_data/procedures.csv,Synthea
2016-07-18T08:20:52.000Z,2016-07-18T08:35:52.000Z,c1f1fcaa-82fd-d5b7-3544-c8f9708b06a8,0bda12c9-bd88-a50f-57e5-9f4245b78b66,430193006,Medication Reconciliation (procedure),805.23,null,null,2026-11-18T13:22:19.610Z,/Volumes/workspace/default/healthcare_source_data/procedures.csv,Synthea


In [0]:
from pyspark.sql.functions import (
    col, to_timestamp, sum as spark_sum, when
)

# Load required tables
bronze_procedures = bronze["procedures"]

silver_procedures = spark.table(
    "workspace.default.healthcare_silver_procedures"
)

silver_patients = spark.table(
    "workspace.default.healthcare_silver_patients"
)

silver_encounters = spark.table(
    "workspace.default.healthcare_silver_encounters"
)

# --------------------------------------------------
# 1. Bronze metrics - calculated in a single aggregation
# --------------------------------------------------

bronze_metrics = bronze_procedures.agg(
    spark_sum(when(col("START").isNotNull() &
                    to_timestamp(col("START")).isNull(), 1)
             .otherwise(0)).alias("start_failures"),

    spark_sum(when(col("STOP").isNotNull() &
                    to_timestamp(col("STOP")).isNull(), 1)
             .otherwise(0)).alias("stop_failures"),

    spark_sum(when(col("BASE_COST").isNotNull() &
                    col("BASE_COST").cast("decimal(18,2)").isNull(), 1)
             .otherwise(0)).alias("cost_failures"),

    spark_sum(when(col("START").isNotNull(), 1)
             .otherwise(0)).alias("bronze_records")
).first()

bronze_count = bronze_metrics["bronze_records"]

# --------------------------------------------------
# 2. Silver record count
# --------------------------------------------------

silver_count = silver_procedures.count()

# --------------------------------------------------
# 3. Timestamp conversion failures
# --------------------------------------------------

timestamp_failures = (
    bronze_metrics["start_failures"] +
    bronze_metrics["stop_failures"]
)

# --------------------------------------------------
# 4. Numeric conversion failures
# --------------------------------------------------

cost_conversion_failures = bronze_metrics["cost_failures"]

# --------------------------------------------------
# 5. Invalid procedure durations
# --------------------------------------------------

invalid_durations = silver_procedures.filter(
    F.col("procedure_start_datetime").isNotNull() &
    F.col("procedure_end_datetime").isNotNull() &
    (F.col("procedure_end_datetime") < F.col("procedure_start_datetime"))
).count()
# --------------------------------------------------
# 6. Orphan patient references
# --------------------------------------------------

orphan_patients = silver_procedures.join(
    silver_patients.select("patient_id"),
    "patient_id",
    "left_anti"
).count()

# --------------------------------------------------
# 7. Orphan encounter references
# --------------------------------------------------

orphan_encounters = silver_procedures.join(
    silver_encounters.select("encounter_id"),
    "encounter_id",
    "left_anti"
).count()

# --------------------------------------------------
# 8. Schema verification
# --------------------------------------------------

expected_schema = {
    "procedure_start_datetime": "timestamp",
    "procedure_stop_datetime": "timestamp",
    "patient_id": "string",
    "encounter_id": "string",
    "procedure_code": "string",
    "procedure_description": "string",
    "base_cost": "decimal(18,2)",
    "reason_code": "string",
    "reason_description": "string",
    "_ingested_at": "timestamp",
    "_source_file": "string",
    "_source_system": "string"
}

actual_schema = {
    field.name: field.dataType.simpleString()
    for field in silver_procedures.schema.fields
}

schema_check = all(
    actual_schema.get(name) == datatype
    for name, datatype in expected_schema.items()
)

# --------------------------------------------------
# 9. Metadata preservation
# --------------------------------------------------

metadata_columns = [
    "_ingested_at",
    "_source_file",
    "_source_system"
]

metadata_check = all(
    name in silver_procedures.columns
    for name in metadata_columns
)

# --------------------------------------------------
# 10. Final validation report
# --------------------------------------------------

validation_results = [
    (
        "Bronze vs Silver Record Count",
        f"Bronze={bronze_count}, Silver={silver_count}",
        "PASS" if bronze_count == silver_count else "FAIL"
    ),
    (
        "Timestamp Conversion Failures",
        timestamp_failures,
        "PASS" if timestamp_failures == 0 else "FAIL"
    ),
    (
        "Numeric Conversion Failures",
        cost_conversion_failures,
        "PASS" if cost_conversion_failures == 0 else "FAIL"
    ),
    (
        "Invalid Procedure Durations",
        invalid_durations,
        "PASS" if invalid_durations == 0 else "FAIL"
    ),
    (
        "Orphan Patient References",
        orphan_patients,
        "PASS" if orphan_patients == 0 else "FAIL"
    ),
    (
        "Orphan Encounter References",
        orphan_encounters,
        "PASS" if orphan_encounters == 0 else "FAIL"
    ),
    (
        "Schema Verification",
        "Schema matches expected types" if schema_check else "Schema mismatch",
        "PASS" if schema_check else "FAIL"
    ),
    (
        "Metadata Preservation",
        "All lineage columns present" if metadata_check else "Missing metadata",
        "PASS" if metadata_check else "FAIL"
    )
]

validation_report = spark.createDataFrame(
    validation_results,
    ["Validation_Check", "Result", "Status"]
)

display(validation_report)

Validation_Check,Result,Status
Bronze vs Silver Record Count,"Bronze=83823, Silver=83823",PASS
Timestamp Conversion Failures,0,PASS
Numeric Conversion Failures,0,PASS
Invalid Procedure Durations,0,PASS
Orphan Patient References,0,PASS
Orphan Encounter References,0,PASS
Schema Verification,Schema mismatch,FAIL
Metadata Preservation,All lineage columns present,PASS


In [0]:
from pyspark.sql import functions as F

# ============================================================
# GOLD TABLE: gold_fact_encounters
# Grain: 1 row per encounter
# Source: Silver Encounters + Silver Patients
# ============================================================

gold_fact_encounters = (
    silver_encounters.alias("e")
    .join(
        silver_patients.alias("p"),
        F.col("e.patient_id") == F.col("p.patient_id"),
        "left"
    )
    .select(
        # Primary identifiers
        F.col("e.encounter_id"),
        F.col("e.patient_id"),
        F.col("e.organization_id"),
        F.col("e.provider_id"),
        F.col("e.payer_id"),

        # Encounter information
        F.col("e.encounter_class"),
        F.col("e.encounter_code"),
        F.col("e.encounter_description"),
        F.col("e.reason_code"),
        F.col("e.reason_description"),

        # Encounter timestamps
        F.col("e.start_datetime"),
        F.col("e.stop_datetime"),

        # Date dimensions
        F.year(F.col("e.start_datetime")).alias("encounter_year"),
        F.month(F.col("e.start_datetime")).alias("encounter_month"),
        F.date_format(F.col("e.start_datetime"), "yyyy-MM").alias("encounter_year_month"),

        # Encounter duration in hours
        F.round(
            (
                F.col("e.stop_datetime").cast("long")
                - F.col("e.start_datetime").cast("long")
            ) / 3600.0,
            2
        ).alias("encounter_duration_hours"),

        # Patient age at encounter
        F.floor(
            F.months_between(
                F.col("e.start_datetime"),
                F.col("p.birth_date")
            ) / 12
        ).alias("patient_age_at_encounter"),

        # Patient demographics
        F.col("p.gender"),
        F.col("p.race"),
        F.col("p.ethnicity"),
        F.col("p.city").alias("patient_city"),
        F.col("p.state").alias("patient_state"),

        # Financial metrics
        F.col("e.base_encounter_cost"),
        F.col("e.total_claim_cost"),
        F.col("e.payer_coverage"),

        # Estimated patient responsibility
        F.round(
            F.col("e.total_claim_cost") - F.col("e.payer_coverage"),
            2
        ).alias("patient_out_of_pocket_estimate"),

        # Data lineage
        F.col("e._ingested_at"),
        F.col("e._source_file"),
        F.col("e._source_system")
    )
)

print("Gold Fact Encounters created successfully.")
display(gold_fact_encounters.limit(5))


Gold Fact Encounters created successfully.


encounter_id,patient_id,organization_id,provider_id,payer_id,encounter_class,encounter_code,encounter_description,reason_code,reason_description,start_datetime,stop_datetime,encounter_year,encounter_month,encounter_year_month,encounter_duration_hours,patient_age_at_encounter,gender,race,ethnicity,patient_city,patient_state,base_encounter_cost,total_claim_cost,payer_coverage,patient_out_of_pocket_estimate,_ingested_at,_source_file,_source_system
748f8357-6cc7-551d-f31a-32fa2cf84126,b9c610cd-28a6-4636-ccb6-c7a0d2a4cb85,f7ae497d-8dc6-3721-9402-43b621a4e7d2,82608ebb-037c-3cef-9d34-3736d69b29e8,7c4411ce-02f1-39b5-b9ec-dfbea9ad3c1a,wellness,410620009,Well child visit (procedure),null,null,2019-02-17T05:07:38.000Z,2019-02-17T05:22:38.000Z,2019,2,2019-02,0.25,0,M,white,nonhispanic,Springfield,Massachusetts,129.16,877.79,833.90,43.89,2026-11-18T13:22:14.232Z,/Volumes/workspace/default/healthcare_source_data/encounters.csv,Synthea
5a4735ae-423f-6563-28ab-b3d11b49b2d4,b9c610cd-28a6-4636-ccb6-c7a0d2a4cb85,f7ae497d-8dc6-3721-9402-43b621a4e7d2,82608ebb-037c-3cef-9d34-3736d69b29e8,7c4411ce-02f1-39b5-b9ec-dfbea9ad3c1a,wellness,410620009,Well child visit (procedure),null,null,2019-03-24T05:07:38.000Z,2019-03-24T05:22:38.000Z,2019,3,2019-03,0.25,0,M,white,nonhispanic,Springfield,Massachusetts,129.16,269.68,256.20,13.48,2026-11-18T13:22:14.232Z,/Volumes/workspace/default/healthcare_source_data/encounters.csv,Synthea
0bee1ce6-3e2c-5506-f71c-a7ba8f64a3d3,b9c610cd-28a6-4636-ccb6-c7a0d2a4cb85,f7ae497d-8dc6-3721-9402-43b621a4e7d2,82608ebb-037c-3cef-9d34-3736d69b29e8,7c4411ce-02f1-39b5-b9ec-dfbea9ad3c1a,wellness,410620009,Well child visit (procedure),null,null,2019-05-26T05:07:38.000Z,2019-05-26T05:22:38.000Z,2019,5,2019-05,0.25,0,M,white,nonhispanic,Springfield,Massachusetts,129.16,1292.75,1228.11,64.64,2026-11-18T13:22:14.232Z,/Volumes/workspace/default/healthcare_source_data/encounters.csv,Synthea
6e93bcf9-45a4-8528-0120-1c1eaa930faf,b9c610cd-28a6-4636-ccb6-c7a0d2a4cb85,f7ae497d-8dc6-3721-9402-43b621a4e7d2,82608ebb-037c-3cef-9d34-3736d69b29e8,7c4411ce-02f1-39b5-b9ec-dfbea9ad3c1a,wellness,410620009,Well child visit (procedure),null,null,2019-07-28T05:07:38.000Z,2019-07-28T05:22:38.000Z,2019,7,2019-07,0.25,0,M,white,nonhispanic,Springfield,Massachusetts,129.16,1323.87,1257.68,66.19,2026-11-18T13:22:14.232Z,/Volumes/workspace/default/healthcare_source_data/encounters.csv,Synthea
8b6787c3-4316-a0cb-899d-4746525c319f,b9c610cd-28a6-4636-ccb6-c7a0d2a4cb85,f7ae497d-8dc6-3721-9402-43b621a4e7d2,82608ebb-037c-3cef-9d34-3736d69b29e8,7c4411ce-02f1-39b5-b9ec-dfbea9ad3c1a,wellness,410620009,Well child visit (procedure),null,null,2019-10-27T05:07:38.000Z,2019-10-27T05:22:38.000Z,2019,10,2019-10,0.25,0,M,white,nonhispanic,Springfield,Massachusetts,129.16,831.76,790.17,41.59,2026-11-18T13:22:14.232Z,/Volumes/workspace/default/healthcare_source_data/encounters.csv,Synthea


In [0]:
gold_fact_encounters.printSchema()

gold_fact_encounters.show(10, truncate=False)

print("Total Gold Encounters:", gold_fact_encounters.count())

root
 |-- encounter_id: string (nullable = true)
 |-- patient_id: string (nullable = true)
 |-- organization_id: string (nullable = true)
 |-- provider_id: string (nullable = true)
 |-- payer_id: string (nullable = true)
 |-- encounter_class: string (nullable = true)
 |-- encounter_code: string (nullable = true)
 |-- encounter_description: string (nullable = true)
 |-- reason_code: string (nullable = true)
 |-- reason_description: string (nullable = true)
 |-- start_datetime: timestamp (nullable = true)
 |-- stop_datetime: timestamp (nullable = true)
 |-- encounter_year: integer (nullable = true)
 |-- encounter_month: integer (nullable = true)
 |-- encounter_year_month: string (nullable = true)
 |-- encounter_duration_hours: double (nullable = true)
 |-- patient_age_at_encounter: long (nullable = true)
 |-- gender: string (nullable = true)
 |-- race: string (nullable = true)
 |-- ethnicity: string (nullable = true)
 |-- patient_city: string (nullable = true)
 |-- patient_state: string 

In [0]:
gold_fact_encounters.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("gold_fact_encounters")

print("gold_fact_encounters saved successfully.")

gold_fact_encounters saved successfully.


In [0]:
from pyspark.sql import functions as F

# =========================================================
# GOLD: PATIENT SUMMARY
# Grain: One row per patient
# Source: gold_fact_encounters
# =========================================================

gold_patient_summary = (
    gold_fact_encounters
    .groupBy("patient_id")
    .agg(
        # Encounter metrics
        F.count("encounter_id").alias("total_encounters"),
        F.countDistinct("encounter_class").alias("distinct_encounter_classes"),

        # Cost metrics
        F.round(F.sum("total_claim_cost"), 2).alias("total_healthcare_cost"),
        F.round(F.sum("payer_coverage"), 2).alias("total_insurance_coverage"),
        F.round(F.sum("patient_out_of_pocket_estimate"), 2).alias("total_out_of_pocket_estimate"),
        F.round(F.avg("total_claim_cost"), 2).alias("average_encounter_cost"),

        # Duration metrics
        F.round(F.sum("encounter_duration_hours"), 2).alias("total_encounter_hours"),
        F.round(F.avg("encounter_duration_hours"), 2).alias("average_encounter_hours"),

        # Visit timeline
        F.min("start_datetime").alias("first_encounter_date"),
        F.max("start_datetime").alias("last_encounter_date")
    )
)

print("Gold Patient Summary created successfully.")

Gold Patient Summary created successfully.


In [0]:
gold_patient_summary.printSchema()

gold_patient_summary.show(10, truncate=False)

print("Total Patients:", gold_patient_summary.count())

root
 |-- patient_id: string (nullable = true)
 |-- total_encounters: long (nullable = false)
 |-- distinct_encounter_classes: long (nullable = false)
 |-- total_healthcare_cost: decimal(29,2) (nullable = true)
 |-- total_insurance_coverage: decimal(29,2) (nullable = true)
 |-- total_out_of_pocket_estimate: decimal(31,2) (nullable = true)
 |-- average_encounter_cost: decimal(19,2) (nullable = true)
 |-- total_encounter_hours: double (nullable = true)
 |-- average_encounter_hours: double (nullable = true)
 |-- first_encounter_date: timestamp (nullable = true)
 |-- last_encounter_date: timestamp (nullable = true)

+------------------------------------+----------------+--------------------------+---------------------+------------------------+----------------------------+----------------------+---------------------+-----------------------+--------------------+-------------------+
|patient_id                          |total_encounters|distinct_encounter_classes|total_healthcare_cost|total_i

In [0]:
from pyspark.sql import functions as F

gold_patient_summary.select(
    "patient_id",
    "total_healthcare_cost",
    "total_insurance_coverage",
    "total_out_of_pocket_estimate",
    F.round(
        F.col("total_healthcare_cost") - F.col("total_insurance_coverage"),
        2
    ).alias("expected_out_of_pocket"),
    F.round(
        F.col("total_out_of_pocket_estimate") -
        (F.col("total_healthcare_cost") - F.col("total_insurance_coverage")),
        2
    ).alias("reconciliation_difference")
).show(10, truncate=False)

+------------------------------------+---------------------+------------------------+----------------------------+----------------------+-------------------------+
|patient_id                          |total_healthcare_cost|total_insurance_coverage|total_out_of_pocket_estimate|expected_out_of_pocket|reconciliation_difference|
+------------------------------------+---------------------+------------------------+----------------------------+----------------------+-------------------------+
|2f031d4a-b070-ce15-6372-30c8fecf1164|460929.49            |223534.14               |237395.35                   |237395.35             |0.00                     |
|4f5a07ca-02e0-8981-3c30-4d9924a169a3|9623.42              |309.70                  |9313.72                     |9313.72               |0.00                     |
|afd63046-5213-0c98-561f-12e6725cd1f5|105269.81            |26644.71                |78625.10                    |78625.10              |0.00                     |
|04f25f73-04b2-4

In [0]:
gold_patient_summary.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("gold_patient_summary")

print("gold_patient_summary saved successfully.")

gold_patient_summary saved successfully.


In [0]:
from pyspark.sql import functions as F

# Source: Silver Conditions
conditions = silver_conditions

# Calculate duration only when both dates exist and duration is non-negative
conditions_with_duration = conditions.withColumn(
    "condition_duration_days",
    F.when(
        F.col("condition_start_date").isNotNull() &
        F.col("condition_end_date").isNotNull() &
        (F.datediff("condition_end_date", "condition_start_date") >= 0),
        F.datediff("condition_end_date", "condition_start_date")
    )
)

# Aggregate condition-level metrics
gold_condition_summary = (
    conditions_with_duration
    .groupBy("condition_code", "condition_description")
    .agg(
        F.count("*").alias("total_condition_records"),
        F.countDistinct("patient_id").alias("unique_patients_affected"),
        F.min("condition_start_date").alias("first_recorded_date"),
        F.max("condition_start_date").alias("last_recorded_date"),
        F.sum(
            F.when(F.col("condition_end_date").isNotNull(), 1).otherwise(0)
        ).alias("records_with_end_date"),
        F.sum(
            F.when(F.col("condition_end_date").isNull(), 1).otherwise(0)
        ).alias("records_without_end_date"),
        F.round(
            F.avg("condition_duration_days"), 2
        ).alias("average_condition_duration_days")
    )
)

gold_condition_summary.show(10, truncate=False)
print("Gold Condition Summary rows:", gold_condition_summary.count())

+--------------+---------------------------------------------------------+-----------------------+------------------------+-------------------+------------------+---------------------+------------------------+-------------------------------+
|condition_code|condition_description                                    |total_condition_records|unique_patients_affected|first_recorded_date|last_recorded_date|records_with_end_date|records_without_end_date|average_condition_duration_days|
+--------------+---------------------------------------------------------+-----------------------+------------------------+-------------------+------------------+---------------------+------------------------+-------------------------------+
|10509002      |Acute bronchitis (disorder)                              |571                    |454                     |1938-01-08         |2021-10-23        |569                  |2                       |13.45                          |
|283371005     |Laceration of fo

In [0]:
# 1. Record reconciliation
silver_count = silver_conditions.count()

gold_record_count = gold_condition_summary.agg(
    F.sum("total_condition_records")
).first()[0]

# 2. Duplicate aggregation keys
duplicate_groups = (
    gold_condition_summary
    .groupBy("condition_code", "condition_description")
    .count()
    .filter(F.col("count") > 1)
    .count()
)

# 3. Records with end dates must reconcile
records_with_end = gold_condition_summary.agg(
    F.sum("records_with_end_date")
).first()[0]

records_without_end = gold_condition_summary.agg(
    F.sum("records_without_end_date")
).first()[0]

# 4. Validate duration aggregation
invalid_duration_count = conditions_with_duration.filter(
    F.col("condition_start_date").isNotNull() &
    F.col("condition_end_date").isNotNull() &
    (F.datediff("condition_end_date", "condition_start_date") < 0)
).count()

# Display focused validation results
validation_results = [
    ("Silver vs Gold Condition Record Count",
     silver_count == gold_record_count),

    ("Duplicate Condition Groups",
     duplicate_groups == 0),

    ("End Date Record Reconciliation",
     records_with_end + records_without_end == silver_count),

    ("Invalid Condition Durations",
     invalid_duration_count == 0)
]

for check_name, passed in validation_results:
    print(f"{check_name}: {'PASS' if passed else 'FAIL'}")

# Stop execution if any validation fails
if not all(passed for _, passed in validation_results):
    raise Exception("Gold Condition Summary validation failed. Table not saved.")

Silver vs Gold Condition Record Count: PASS
Duplicate Condition Groups: PASS
End Date Record Reconciliation: PASS
Invalid Condition Durations: PASS


In [0]:

gold_condition_summary.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("workspace.default.gold_condition_summary")

print("Gold Condition Summary successfully saved.")


Gold Condition Summary successfully saved.


In [0]:
from pyspark.sql import functions as F

# Source: validated Silver Procedures
procedures = silver_procedures

# Calculate procedure duration in hours
procedures_with_duration = procedures.withColumn(
    "procedure_duration_hours",
    F.when(
        F.col("procedure_start_datetime").isNotNull() &
        F.col("procedure_end_datetime").isNotNull() &
        (F.col("procedure_end_datetime") >= F.col("procedure_start_datetime")),
        (
            F.unix_timestamp("procedure_end_datetime") -
            F.unix_timestamp("procedure_start_datetime")
        ) / 3600
    )
)

# Gold aggregation at procedure level
gold_procedure_summary = (
    procedures_with_duration
    .groupBy("procedure_code", "procedure_description")
    .agg(
        F.count("*").alias("total_procedures"),
        F.countDistinct("patient_id").alias("unique_patients"),
        F.round(F.sum("base_cost"), 2).alias("total_procedure_cost"),
        F.round(F.avg("base_cost"), 2).alias("average_procedure_cost"),
        F.round(F.sum("procedure_duration_hours"), 2).alias("total_procedure_hours"),
        F.round(F.avg("procedure_duration_hours"), 2).alias("average_procedure_hours"),
        F.min("procedure_start_datetime").alias("first_procedure_datetime"),
        F.max("procedure_start_datetime").alias("last_procedure_datetime")
    )
)

gold_procedure_summary.show(10, truncate=False)

print("Gold Procedure Summary rows:", gold_procedure_summary.count())

+--------------+-----------------------------------------------------------------------------------+----------------+---------------+--------------------+----------------------+---------------------+-----------------------+------------------------+-----------------------+
|procedure_code|procedure_description                                                              |total_procedures|unique_patients|total_procedure_cost|average_procedure_cost|total_procedure_hours|average_procedure_hours|first_procedure_datetime|last_procedure_datetime|
+--------------+-----------------------------------------------------------------------------------+----------------+---------------+--------------------+----------------------+---------------------+-----------------------+------------------------+-----------------------+
|430193006     |Medication Reconciliation (procedure)                                              |6032            |1145           |3628579.16          |601.55                |1503

In [0]:
# Record reconciliation
silver_count = silver_procedures.count()

gold_count = gold_procedure_summary.agg(
    F.sum("total_procedures")
).first()[0]

# Check duplicate aggregation keys
duplicate_groups = (
    gold_procedure_summary
    .groupBy("procedure_code", "procedure_description")
    .count()
    .filter(F.col("count") > 1)
    .count()
)

# Check invalid procedure durations
# Check invalid procedure durations
invalid_duration_count = procedures.filter(
    F.col("procedure_start_datetime").isNotNull() &
    F.col("procedure_end_datetime").isNotNull() &
    (F.col("procedure_end_datetime") < F.col("procedure_start_datetime"))
).count()

# Validation results
procedure_checks = [
    ("Silver vs Gold Procedure Count", silver_count == gold_count),
    ("Duplicate Procedure Groups", duplicate_groups == 0),
    ("Invalid Procedure Durations", invalid_duration_count == 0)
]

for check, passed in procedure_checks:
    print(f"{check}: {'PASS' if passed else 'FAIL'}")

if not all(passed for _, passed in procedure_checks):
    raise Exception("Gold Procedure Summary validation failed. Table not saved.")

Silver vs Gold Procedure Count: PASS
Duplicate Procedure Groups: PASS
Invalid Procedure Durations: PASS


In [0]:
gold_procedure_summary.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("workspace.default.gold_procedure_summary")

In [0]:
saved_gold_procedures = spark.table(
    "workspace.default.gold_procedure_summary"
)

saved_gold_procedures.show(10, truncate=False)

print("Total Gold Procedure Summary rows:", saved_gold_procedures.count())

+--------------+-------------------------------------------------------+----------------+---------------+--------------------+----------------------+---------------------+-----------------------+------------------------+-----------------------+
|procedure_code|procedure_description                                  |total_procedures|unique_patients|total_procedure_cost|average_procedure_cost|total_procedure_hours|average_procedure_hours|first_procedure_datetime|last_procedure_datetime|
+--------------+-------------------------------------------------------+----------------+---------------+--------------------+----------------------+---------------------+-----------------------+------------------------+-----------------------+
|387607004     |Construction of diverting colostomy                    |1               |1              |55863.56            |55863.56              |0.25                 |0.25                   |2006-02-18 21:56:43     |2006-02-18 21:56:43    |
|261352009     |Face

In [0]:
'''Metrics:
- Total encounters handled
- Unique patients served
- Unique providers
- Encounter class diversity
- Total healthcare cost
- Total claim cost
- Total payer coverage
- Average encounter cost
- First and last encounter dates'''
from pyspark.sql import functions as F

# Source: validated Silver Encounters
encounters = silver_encounters

gold_organization_summary = (
    encounters
    .groupBy("organization_id")
    .agg(
        F.count("*").alias("total_encounters"),
        F.countDistinct("patient_id").alias("unique_patients_served"),
        F.countDistinct("provider_id").alias("unique_providers"),
        F.countDistinct("encounter_class").alias("distinct_encounter_classes"),
        F.round(F.sum("base_encounter_cost"), 2).alias("total_base_encounter_cost"),
        F.round(F.sum("total_claim_cost"), 2).alias("total_claim_cost"),
        F.round(F.sum("payer_coverage"), 2).alias("total_payer_coverage"),
        F.round(F.avg("total_claim_cost"), 2).alias("average_claim_cost"),
        F.min("start_datetime").alias("first_encounter_datetime"),
        F.max("start_datetime").alias("last_encounter_datetime")
    )
)

gold_organization_summary.show(10, truncate=False)

print("Gold Organization Summary rows:",
      gold_organization_summary.count())

+------------------------------------+----------------+----------------------+----------------+--------------------------+-------------------------+----------------+--------------------+------------------+------------------------+-----------------------+
|organization_id                     |total_encounters|unique_patients_served|unique_providers|distinct_encounter_classes|total_base_encounter_cost|total_claim_cost|total_payer_coverage|average_claim_cost|first_encounter_datetime|last_encounter_datetime|
+------------------------------------+----------------+----------------------+----------------+--------------------------+-------------------------+----------------+--------------------+------------------+------------------------+-----------------------+
|f7ae497d-8dc6-3721-9402-43b621a4e7d2|10              |1                     |1               |1                         |1291.60                  |7831.66         |7440.08             |783.17            |2019-02-17 05:07:38     |2021-

In [0]:
silver_count = silver_encounters.count()

gold_count = gold_organization_summary.agg(
    F.sum("total_encounters")
).first()[0]

duplicate_organizations = (
    gold_organization_summary
    .groupBy("organization_id")
    .count()
    .filter(F.col("count") > 1)
    .count()
)

checks = [
    ("Silver vs Gold Encounter Reconciliation",
     silver_count == gold_count),

    ("Duplicate Organization Keys",
     duplicate_organizations == 0)
]

for check_name, passed in checks:
    print(f"{check_name}: {'PASS' if passed else 'FAIL'}")

if not all(passed for _, passed in checks):
    raise Exception("Gold Organization Summary validation failed.")

Silver vs Gold Encounter Reconciliation: PASS
Duplicate Organization Keys: PASS


In [0]:
# Financial reconciliation
silver_claim_total = silver_encounters.agg(
    F.round(F.sum("total_claim_cost"), 2)
).first()[0] or 0

gold_claim_total = gold_organization_summary.agg(
    F.round(F.sum("total_claim_cost"), 2)
).first()[0] or 0

claim_cost_check = abs(silver_claim_total - gold_claim_total) < 0.02

print(
    f"Silver Claim Total: {silver_claim_total} | "
    f"Gold Claim Total: {gold_claim_total} | "
    f"{'PASS' if claim_cost_check else 'FAIL'}"
)

if not claim_cost_check:
    raise Exception("Claim cost reconciliation failed.")

Silver Claim Total: 255033828.08 | Gold Claim Total: 255033828.08 | PASS


In [0]:
gold_organization_summary.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("workspace.default.gold_organization_summary")

print("Gold Organization Summary saved successfully.")

Gold Organization Summary saved successfully.


In [0]:
saved_gold_organizations = spark.table(
    "workspace.default.gold_organization_summary"
)

saved_gold_organizations.show(10, truncate=False)

print(
    "Total Gold Organizations:",
    saved_gold_organizations.count()
)

+------------------------------------+----------------+----------------------+----------------+--------------------------+-------------------------+----------------+--------------------+------------------+------------------------+-----------------------+
|organization_id                     |total_encounters|unique_patients_served|unique_providers|distinct_encounter_classes|total_base_encounter_cost|total_claim_cost|total_payer_coverage|average_claim_cost|first_encounter_datetime|last_encounter_datetime|
+------------------------------------+----------------+----------------------+----------------+--------------------------+-------------------------+----------------+--------------------+------------------+------------------------+-----------------------+
|226098a2-6a40-3588-b5bb-db56c3a30a04|835             |46                    |1               |4                         |85630.50                 |6495903.15      |764516.63           |7779.52           |1943-12-05 03:21:54     |2021-

In [0]:
'''Metrics:
- Total encounters per provider
- Unique patients treated
- Encounter classes handled
- Total base encounter cost
- Total claim cost
- Total payer coverage
- Average claim cost
- First and last encounter date'''
from pyspark.sql import functions as F

# Source: Silver Encounters
encounters = silver_encounters

gold_provider_summary = (
    encounters
    .groupBy("provider_id")
    .agg(
        F.count("*").alias("total_encounters"),
        F.countDistinct("patient_id").alias("unique_patients_treated"),
        F.countDistinct("encounter_class").alias("distinct_encounter_classes"),
        F.round(F.sum("base_encounter_cost"), 2).alias("total_base_encounter_cost"),
        F.round(F.sum("total_claim_cost"), 2).alias("total_claim_cost"),
        F.round(F.sum("payer_coverage"), 2).alias("total_payer_coverage"),
        F.round(F.avg("total_claim_cost"), 2).alias("average_claim_cost"),
        F.min("start_datetime").alias("first_encounter_datetime"),
        F.max("start_datetime").alias("last_encounter_datetime")
    )
)

gold_provider_summary.show(10, truncate=False)

print("Gold Provider Summary rows:", gold_provider_summary.count())

+------------------------------------+----------------+-----------------------+--------------------------+-------------------------+----------------+--------------------+------------------+------------------------+-----------------------+
|provider_id                         |total_encounters|unique_patients_treated|distinct_encounter_classes|total_base_encounter_cost|total_claim_cost|total_payer_coverage|average_claim_cost|first_encounter_datetime|last_encounter_datetime|
+------------------------------------+----------------+-----------------------+--------------------------+-------------------------+----------------+--------------------+------------------+------------------------+-----------------------+
|82608ebb-037c-3cef-9d34-3736d69b29e8|10              |1                      |1                         |1291.60                  |7831.66         |7440.08             |783.17            |2019-02-17 05:07:38     |2021-07-25 05:07:38    |
|9f529022-166f-3db4-9f92-ff1ba11ee565|365   

In [0]:
# Record reconciliation
silver_encounter_count = silver_encounters.count()

gold_encounter_count = gold_provider_summary.agg(
    F.sum("total_encounters")
).first()[0] or 0

# Financial reconciliation
silver_claim_total = silver_encounters.agg(
    F.round(F.sum("total_claim_cost"), 2)
).first()[0] or 0

gold_claim_total = gold_provider_summary.agg(
    F.round(F.sum("total_claim_cost"), 2)
).first()[0] or 0

provider_checks = [
    ("Encounter Record Reconciliation",
     silver_encounter_count == gold_encounter_count),

    ("Claim Cost Reconciliation",
     abs(silver_claim_total - gold_claim_total) < 0.02)
]

for check_name, passed in provider_checks:
    print(f"{check_name}: {'PASS' if passed else 'FAIL'}")

if not all(passed for _, passed in provider_checks):
    raise Exception("Gold Provider Summary validation failed. Table not saved.")

Encounter Record Reconciliation: PASS
Claim Cost Reconciliation: PASS


In [0]:
gold_provider_summary.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("workspace.default.gold_provider_summary")

print("Gold Provider Summary saved successfully.")

Gold Provider Summary saved successfully.


In [0]:
saved_gold_providers = spark.table(
    "workspace.default.gold_provider_summary"
)

saved_gold_providers.show(10, truncate=False)

print("Saved Gold Provider Summary count:",
      saved_gold_providers.count())

+------------------------------------+----------------+-----------------------+--------------------------+-------------------------+----------------+--------------------+------------------+------------------------+-----------------------+
|provider_id                         |total_encounters|unique_patients_treated|distinct_encounter_classes|total_base_encounter_cost|total_claim_cost|total_payer_coverage|average_claim_cost|first_encounter_datetime|last_encounter_datetime|
+------------------------------------+----------------+-----------------------+--------------------------+-------------------------+----------------+--------------------+------------------+------------------------+-----------------------+
|5d575df7-fde0-3dd2-b94d-d03a0212e99b|31              |17                     |1                         |4003.96                  |6533.32         |866.35              |210.75            |1972-02-03 13:18:15     |2021-08-21 10:43:04    |
|320d3d09-ec75-3037-b7db-070cf4ee81f4|11    

In [0]:
'''Metrics:
- Total encounters
- Unique patients covered
- Unique organizations
- Unique providers
- Total claim cost
- Total payer coverage
- Total patient out-of-pocket estimate
- Average claim cost
- Average payer coverage per encounter
- First and last encounter dates'''
from pyspark.sql import functions as F

# Source: validated Silver Encounters
encounters = silver_encounters

gold_payer_summary = (
    encounters
    .groupBy("payer_id")
    .agg(
        F.count("*").alias("total_encounters"),
        F.countDistinct("patient_id").alias("unique_patients"),
        F.countDistinct("organization_id").alias("unique_organizations"),
        F.countDistinct("provider_id").alias("unique_providers"),
        F.round(F.sum("total_claim_cost"), 2).alias("total_claim_cost"),
        F.round(F.sum("payer_coverage"), 2).alias("total_payer_coverage"),
        F.round(
            F.sum(F.col("total_claim_cost") - F.col("payer_coverage")), 2
        ).alias("total_out_of_pocket_estimate"),
        F.round(F.avg("total_claim_cost"), 2).alias("average_claim_cost"),
        F.round(F.avg("payer_coverage"), 2).alias("average_payer_coverage"),
        F.min("start_datetime").alias("first_encounter_datetime"),
        F.max("start_datetime").alias("last_encounter_datetime")
    )
)

gold_payer_summary.show(10, truncate=False)

print("Gold Payer Summary rows:", gold_payer_summary.count())

+------------------------------------+----------------+---------------+--------------------+----------------+----------------+--------------------+----------------------------+------------------+----------------------+------------------------+-----------------------+
|payer_id                            |total_encounters|unique_patients|unique_organizations|unique_providers|total_claim_cost|total_payer_coverage|total_out_of_pocket_estimate|average_claim_cost|average_payer_coverage|first_encounter_datetime|last_encounter_datetime|
+------------------------------------+----------------+---------------+--------------------+----------------+----------------+--------------------+----------------------------+------------------+----------------------+------------------------+-----------------------+
|7c4411ce-02f1-39b5-b9ec-dfbea9ad3c1a|5283            |363            |374                 |374             |25966027.19     |24423140.77         |1542886.42                  |4915.02           |4

In [0]:
# Record reconciliation
silver_count = silver_encounters.count()

gold_count = gold_payer_summary.agg(
    F.sum("total_encounters")
).first()[0] or 0

# Claim cost reconciliation
silver_claim_total = silver_encounters.agg(
    F.round(F.sum("total_claim_cost"), 2)
).first()[0] or 0

gold_claim_total = gold_payer_summary.agg(
    F.round(F.sum("total_claim_cost"), 2)
).first()[0] or 0

# Payer coverage reconciliation
silver_coverage_total = silver_encounters.agg(
    F.round(F.sum("payer_coverage"), 2)
).first()[0] or 0

gold_coverage_total = gold_payer_summary.agg(
    F.round(F.sum("total_payer_coverage"), 2)
).first()[0] or 0

checks = [
    ("Encounter Count Reconciliation", silver_count == gold_count),
    ("Claim Cost Reconciliation", abs(silver_claim_total - gold_claim_total) < 0.02),
    ("Payer Coverage Reconciliation", abs(silver_coverage_total - gold_coverage_total) < 0.02)
]

for check_name, passed in checks:
    print(f"{check_name}: {'PASS' if passed else 'FAIL'}")

if not all(passed for _, passed in checks):
    raise Exception("Gold Payer Summary validation failed. Table not saved.")

Encounter Count Reconciliation: PASS
Claim Cost Reconciliation: PASS
Payer Coverage Reconciliation: PASS


In [0]:
gold_payer_summary.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("workspace.default.gold_payer_summary")

print("Gold Payer Summary saved successfully.")

Gold Payer Summary saved successfully.


In [0]:
saved_gold_payers = spark.table(
    "workspace.default.gold_payer_summary"
)

saved_gold_payers.show(10, truncate=False)

print("Saved Gold Payer Summary count:", saved_gold_payers.count())

+------------------------------------+----------------+---------------+--------------------+----------------+----------------+--------------------+----------------------------+------------------+----------------------+------------------------+-----------------------+
|payer_id                            |total_encounters|unique_patients|unique_organizations|unique_providers|total_claim_cost|total_payer_coverage|total_out_of_pocket_estimate|average_claim_cost|average_payer_coverage|first_encounter_datetime|last_encounter_datetime|
+------------------------------------+----------------+---------------+--------------------+----------------+----------------+--------------------+----------------------------+------------------+----------------------+------------------------+-----------------------+
|5059a55e-5d6e-34d1-b6cb-d83d16e57bcf|5247            |862            |806                 |807             |13851239.73     |4643.45             |13846596.28                 |2639.84           |0

In [0]:
from pyspark.sql import functions as F

# Expected Gold tables
gold_tables = [
    "gold_fact_encounters",
    "gold_patient_summary",
    "gold_condition_summary",
    "gold_procedure_summary",
    "gold_organization_summary",
    "gold_provider_summary",
    "gold_payer_summary"
]

# Check table existence
existing_tables = {
    row["tableName"]
    for row in spark.sql("SHOW TABLES IN workspace.default").collect()
}

missing_tables = [t for t in gold_tables if t not in existing_tables]

print("Missing Gold Tables:", missing_tables)

if missing_tables:
    raise Exception(f"Gold tables missing: {missing_tables}")

print("All expected Gold tables exist.")

Missing Gold Tables: []
All expected Gold tables exist.


In [0]:
# Source counts
silver_encounter_count = silver_encounters.count()
silver_patient_count = silver_patients.count()
silver_condition_count = silver_conditions.count()
silver_procedure_count = silver_procedures.count()

# Gold counts / totals
gold_fact_count = spark.table(
    "workspace.default.gold_fact_encounters"
).count()

gold_patient_count = spark.table(
    "workspace.default.gold_patient_summary"
).count()

gold_condition_total = spark.table(
    "workspace.default.gold_condition_summary"
).agg(
    F.sum("total_condition_records")
).first()[0]

gold_procedure_total = spark.table(
    "workspace.default.gold_procedure_summary"
).agg(
    F.sum("total_procedures")
).first()[0]

gold_org_encounters = spark.table(
    "workspace.default.gold_organization_summary"
).agg(
    F.sum("total_encounters")
).first()[0]

gold_provider_encounters = spark.table(
    "workspace.default.gold_provider_summary"
).agg(
    F.sum("total_encounters")
).first()[0]

gold_payer_encounters = spark.table(
    "workspace.default.gold_payer_summary"
).agg(
    F.sum("total_encounters")
).first()[0]

# Final reconciliation
final_checks = [
    ("Encounter Fact Count", silver_encounter_count == gold_fact_count),
    ("Patient Summary Count", silver_patient_count == gold_patient_count),
    ("Condition Record Reconciliation", silver_condition_count == gold_condition_total),
    ("Procedure Record Reconciliation", silver_procedure_count == gold_procedure_total),
    ("Organization Encounter Reconciliation", silver_encounter_count == gold_org_encounters),
    ("Provider Encounter Reconciliation", silver_encounter_count == gold_provider_encounters),
    ("Payer Encounter Reconciliation", silver_encounter_count == gold_payer_encounters)
]

for check_name, passed in final_checks:
    print(f"{check_name}: {'PASS' if passed else 'FAIL'}")

if not all(passed for _, passed in final_checks):
    raise Exception("Final Gold Layer Reconciliation Failed")

print("FINAL GOLD LAYER VALIDATION: PASS")

Encounter Fact Count: PASS
Patient Summary Count: PASS
Condition Record Reconciliation: PASS
Procedure Record Reconciliation: PASS
Organization Encounter Reconciliation: PASS
Provider Encounter Reconciliation: PASS
Payer Encounter Reconciliation: PASS
FINAL GOLD LAYER VALIDATION: PASS


In [0]:
from pyspark.sql import functions as F

# ============================================================
# GOLD PRIMARY KEY AUDIT - CORRECTED COMPOSITE KEYS
# ============================================================

gold_key_definitions = {
    "gold_fact_encounters": ["encounter_id"],
    "gold_patient_summary": ["patient_id"],
    "gold_condition_summary": ["condition_code", "condition_description"],
    "gold_procedure_summary": ["procedure_code", "procedure_description"],
    "gold_organization_summary": ["organization_id"],
    "gold_provider_summary": ["provider_id"],
    "gold_payer_summary": ["payer_id"]
}

audit_results = []

for table_name, key_columns in gold_key_definitions.items():

    df = spark.table(f"workspace.default.{table_name}")

    # Null key check
    null_condition = None

    for column in key_columns:
        condition = F.col(column).isNull()

        null_condition = (
            condition
            if null_condition is None
            else null_condition | condition
        )

    null_key_count = df.filter(null_condition).count()

    # Composite-key duplicate check
    duplicate_key_count = (
        df.groupBy(*key_columns)
          .count()
          .filter(F.col("count") > 1)
          .count()
    )

    total_records = df.count()

    status = (
        "PASS"
        if null_key_count == 0 and duplicate_key_count == 0
        else "FAIL"
    )

    audit_results.append(
        (
            table_name,
            ", ".join(key_columns),
            total_records,
            null_key_count,
            duplicate_key_count,
            status
        )
    )

# ============================================================
# DISPLAY AUDIT RESULTS
# ============================================================

gold_primary_key_audit = spark.createDataFrame(
    audit_results,
    [
        "table_name",
        "key_columns",
        "total_records",
        "null_key_count",
        "duplicate_key_count",
        "status"
    ]
)

gold_primary_key_audit.show(truncate=False)

# ============================================================
# FAIL ONLY IF ACTUAL COMPOSITE KEYS ARE INVALID
# ============================================================

failed_checks = (
    gold_primary_key_audit
    .filter(F.col("status") == "FAIL")
    .count()
)

if failed_checks > 0:
    raise Exception("Gold Composite Primary Key Audit FAILED")

print("GOLD COMPOSITE PRIMARY KEY AUDIT: PASS")

+-------------------------+-------------------------------------+-------------+--------------+-------------------+------+
|table_name               |key_columns                          |total_records|null_key_count|duplicate_key_count|status|
+-------------------------+-------------------------------------+-------------+--------------+-------------------+------+
|gold_fact_encounters     |encounter_id                         |61459        |0             |0                  |PASS  |
|gold_patient_summary     |patient_id                           |1163         |0             |0                  |PASS  |
|gold_condition_summary   |condition_code, condition_description|204          |0             |0                  |PASS  |
|gold_procedure_summary   |procedure_code, procedure_description|178          |0             |0                  |PASS  |
|gold_organization_summary|organization_id                      |1122         |0             |0                  |PASS  |
|gold_provider_summary  

In [0]:
from pyspark.sql import functions as F

# ============================================================
# FINAL GOLD LAYER VALIDATION
# Reconciliation + Primary Key Integrity
# Read-only validation; no data writes
# ============================================================

# ------------------------------------------------------------
# 1. SOURCE SILVER COUNTS
# ------------------------------------------------------------

silver_encounter_count = spark.table(
    "workspace.default.healthcare_silver_encounters"
).count()

silver_patient_count = spark.table(
    "workspace.default.healthcare_silver_patients"
).count()

silver_condition_count = spark.table(
    "workspace.default.healthcare_silver_conditions"
).count()

silver_procedure_count = spark.table(
    "workspace.default.healthcare_silver_procedures"
).count()


# ------------------------------------------------------------
# 2. GOLD RECONCILIATION
# ------------------------------------------------------------

gold_fact_count = spark.table(
    "workspace.default.gold_fact_encounters"
).count()

gold_patient_count = spark.table(
    "workspace.default.gold_patient_summary"
).count()

gold_condition_total = spark.table(
    "workspace.default.gold_condition_summary"
).agg(
    F.sum("total_condition_records")
).first()[0]

gold_procedure_total = spark.table(
    "workspace.default.gold_procedure_summary"
).agg(
    F.sum("total_procedures")
).first()[0]

gold_org_encounters = spark.table(
    "workspace.default.gold_organization_summary"
).agg(
    F.sum("total_encounters")
).first()[0]

gold_provider_encounters = spark.table(
    "workspace.default.gold_provider_summary"
).agg(
    F.sum("total_encounters")
).first()[0]

gold_payer_encounters = spark.table(
    "workspace.default.gold_payer_summary"
).agg(
    F.sum("total_encounters")
).first()[0]


# ------------------------------------------------------------
# 3. RECONCILIATION CHECKS
# ------------------------------------------------------------

reconciliation_checks = [
    ("Encounter Fact Count",
     silver_encounter_count == gold_fact_count),

    ("Patient Summary Count",
     silver_patient_count == gold_patient_count),

    ("Condition Record Reconciliation",
     silver_condition_count == gold_condition_total),

    ("Procedure Record Reconciliation",
     silver_procedure_count == gold_procedure_total),

    ("Organization Encounter Reconciliation",
     silver_encounter_count == gold_org_encounters),

    ("Provider Encounter Reconciliation",
     silver_encounter_count == gold_provider_encounters),

    ("Payer Encounter Reconciliation",
     silver_encounter_count == gold_payer_encounters)
]


# ------------------------------------------------------------
# 4. COMPOSITE PRIMARY KEY AUDIT
# ------------------------------------------------------------

gold_key_definitions = {
    "gold_fact_encounters": ["encounter_id"],
    "gold_patient_summary": ["patient_id"],
    "gold_condition_summary": [
        "condition_code",
        "condition_description"
    ],
    "gold_procedure_summary": [
        "procedure_code",
        "procedure_description"
    ],
    "gold_organization_summary": ["organization_id"],
    "gold_provider_summary": ["provider_id"],
    "gold_payer_summary": ["payer_id"]
}

primary_key_results = []

for table_name, key_columns in gold_key_definitions.items():

    df = spark.table(f"workspace.default.{table_name}")

    # Check null primary key values
    null_condition = F.col(key_columns[0]).isNull()

    for column in key_columns[1:]:
        null_condition = null_condition | F.col(column).isNull()

    null_count = df.filter(null_condition).count()

    # Check duplicate composite keys
    duplicate_count = (
        df.groupBy(*key_columns)
          .count()
          .filter(F.col("count") > 1)
          .count()
    )

    pk_pass = null_count == 0 and duplicate_count == 0

    primary_key_results.append(
        (
            table_name,
            null_count,
            duplicate_count,
            pk_pass
        )
    )


# ------------------------------------------------------------
# 5. DISPLAY RECONCILIATION RESULTS
# ------------------------------------------------------------

print("=" * 60)
print("GOLD DATA RECONCILIATION")
print("=" * 60)

for check_name, passed in reconciliation_checks:
    print(f"{check_name}: {'PASS' if passed else 'FAIL'}")


# ------------------------------------------------------------
# 6. DISPLAY PRIMARY KEY AUDIT RESULTS
# ------------------------------------------------------------

pk_audit_df = spark.createDataFrame(
    primary_key_results,
    [
        "table_name",
        "null_key_count",
        "duplicate_key_count",
        "passed"
    ]
)

print("\n" + "=" * 60)
print("GOLD PRIMARY KEY INTEGRITY")
print("=" * 60)

pk_audit_df.show(truncate=False)


# ------------------------------------------------------------
# 7. FINAL VALIDATION GATE
# ------------------------------------------------------------

reconciliation_pass = all(
    passed for _, passed in reconciliation_checks
)

primary_key_pass = (
    pk_audit_df.filter(~F.col("passed")).count() == 0
)

if not reconciliation_pass or not primary_key_pass:
    raise Exception("FINAL GOLD LAYER VALIDATION FAILED")

print("=" * 60)
print("FINAL GOLD LAYER VALIDATION: PASS")
print("ALL RECONCILIATION CHECKS: PASS")
print("ALL PRIMARY KEY CHECKS: PASS")
print("=" * 60)

GOLD DATA RECONCILIATION
Encounter Fact Count: PASS
Patient Summary Count: PASS
Condition Record Reconciliation: PASS
Procedure Record Reconciliation: PASS
Organization Encounter Reconciliation: PASS
Provider Encounter Reconciliation: PASS
Payer Encounter Reconciliation: PASS

GOLD PRIMARY KEY INTEGRITY
+-------------------------+--------------+-------------------+------+
|table_name               |null_key_count|duplicate_key_count|passed|
+-------------------------+--------------+-------------------+------+
|gold_fact_encounters     |0             |0                  |true  |
|gold_patient_summary     |0             |0                  |true  |
|gold_condition_summary   |0             |0                  |true  |
|gold_procedure_summary   |0             |0                  |true  |
|gold_organization_summary|0             |0                  |true  |
|gold_provider_summary    |0             |0                  |true  |
|gold_payer_summary       |0             |0                  |tru

In [0]:
'''Business questions we want to answer
1. How many patients are in our healthcare dataset?
2. How many encounters occurred?
3. How many patients are deceased vs. living?
4. What is the average encounters per patient?
5. What is the total healthcare claim cost?
6. How much was covered by payers?
7. What is the estimated patient responsibility?
8. How many organizations, providers, and payers are represented'''

gold_tables_for_kpi = [
    "gold_fact_encounters",
    "gold_patient_summary",
    "gold_organization_summary",
    "gold_provider_summary",
    "gold_payer_summary"
]

for table_name in gold_tables_for_kpi:
    print(f"\n{'=' * 70}")
    print(f"TABLE: workspace.default.{table_name}")
    print("=" * 70)

    spark.table(f"workspace.default.{table_name}").printSchema()




TABLE: workspace.default.gold_fact_encounters
root
 |-- encounter_id: string (nullable = true)
 |-- patient_id: string (nullable = true)
 |-- organization_id: string (nullable = true)
 |-- provider_id: string (nullable = true)
 |-- payer_id: string (nullable = true)
 |-- encounter_class: string (nullable = true)
 |-- encounter_code: string (nullable = true)
 |-- encounter_description: string (nullable = true)
 |-- reason_code: string (nullable = true)
 |-- reason_description: string (nullable = true)
 |-- start_datetime: timestamp (nullable = true)
 |-- stop_datetime: timestamp (nullable = true)
 |-- encounter_year: integer (nullable = true)
 |-- encounter_month: integer (nullable = true)
 |-- encounter_year_month: string (nullable = true)
 |-- encounter_duration_hours: double (nullable = true)
 |-- patient_age_at_encounter: long (nullable = true)
 |-- gender: string (nullable = true)
 |-- race: string (nullable = true)
 |-- ethnicity: string (nullable = true)
 |-- patient_city: strin

In [0]:
patient_encounter_kpi = spark.sql("""
WITH patient_metrics AS (
    SELECT
        COUNT(*) AS total_patients,
        SUM(CASE WHEN death_date IS NULL THEN 1 ELSE 0 END) AS living_patients,
        SUM(CASE WHEN death_date IS NOT NULL THEN 1 ELSE 0 END) AS deceased_patients
    FROM workspace.default.healthcare_silver_patients
),

encounter_metrics AS (
    SELECT
        COUNT(*) AS total_encounters,
        COUNT(DISTINCT patient_id) AS patients_with_encounters,
        ROUND(SUM(total_claim_cost), 2) AS total_claim_cost,
        ROUND(SUM(payer_coverage), 2) AS total_payer_coverage,
        ROUND(SUM(patient_out_of_pocket_estimate), 2) AS total_patient_out_of_pocket,
        ROUND(AVG(total_claim_cost), 2) AS average_claim_cost_per_encounter
    FROM workspace.default.gold_fact_encounters
),

organization_metrics AS (
    SELECT COUNT(*) AS total_organizations
    FROM workspace.default.gold_organization_summary
),

provider_metrics AS (
    SELECT COUNT(*) AS total_providers
    FROM workspace.default.gold_provider_summary
),

payer_metrics AS (
    SELECT COUNT(*) AS total_payers
    FROM workspace.default.gold_payer_summary
)

SELECT
    p.total_patients,
    p.living_patients,
    p.deceased_patients,
    e.total_encounters,
    e.patients_with_encounters,
    ROUND(
        e.total_encounters / NULLIF(e.patients_with_encounters, 0),
        2
    ) AS average_encounters_per_patient,
    o.total_organizations,
    pr.total_providers,
    py.total_payers,
    e.total_claim_cost,
    e.total_payer_coverage,
    e.total_patient_out_of_pocket,
    e.average_claim_cost_per_encounter
FROM patient_metrics p
CROSS JOIN encounter_metrics e
CROSS JOIN organization_metrics o
CROSS JOIN provider_metrics pr
CROSS JOIN payer_metrics py
""")

display(patient_encounter_kpi)

total_patients,living_patients,deceased_patients,total_encounters,patients_with_encounters,average_encounters_per_patient,total_organizations,total_providers,total_payers,total_claim_cost,total_payer_coverage,total_patient_out_of_pocket,average_claim_cost_per_encounter
1163,1000,163,61459,1163,52.85,1122,1123,10,255033828.08,63530758.42,191503069.66,4149.66


In [0]:
kpi_result = patient_encounter_kpi.first()

# Patient count reconciliation
patient_count_check = (
    kpi_result["total_patients"]
    == kpi_result["living_patients"] + kpi_result["deceased_patients"]
)

# Encounter count reconciliation
encounter_count_check = (
    kpi_result["total_encounters"]
    == spark.table("workspace.default.gold_fact_encounters").count()
)

print("Patient Mortality Reconciliation:",
      "PASS" if patient_count_check else "FAIL")

print("Encounter Count Reconciliation:",
      "PASS" if encounter_count_check else "FAIL")

if not patient_count_check or not encounter_count_check:
    raise Exception("Patient & Encounter KPI validation failed")

print("PATIENT & ENCOUNTER KPI VALIDATION: PASS")

Patient Mortality Reconciliation: PASS
Encounter Count Reconciliation: PASS
PATIENT & ENCOUNTER KPI VALIDATION: PASS


In [0]:
'''We will analyze monthly trends to understand:
- How encounter volume changes over time
- How many unique patients visit each month
- Monthly healthcare claim expenditure
- Insurance coverage per month
- Average claim cost per encounter
- Average encounter duration'''

monthly_encounter_trends = spark.sql("""
SELECT
    encounter_year,
    encounter_month,
    encounter_year_month,

    COUNT(encounter_id) AS total_encounters,

    COUNT(DISTINCT patient_id) AS unique_patients,

    ROUND(SUM(total_claim_cost), 2) AS total_monthly_claim_cost,

    ROUND(SUM(payer_coverage), 2) AS total_monthly_payer_coverage,

    ROUND(SUM(patient_out_of_pocket_estimate), 2) 
        AS total_monthly_patient_out_of_pocket,

    ROUND(AVG(total_claim_cost), 2) 
        AS average_claim_cost_per_encounter,

    ROUND(AVG(encounter_duration_hours), 2) 
        AS average_encounter_duration_hours

FROM workspace.default.gold_fact_encounters

GROUP BY
    encounter_year,
    encounter_month,
    encounter_year_month

ORDER BY
    encounter_year,
    encounter_month
""")

display(monthly_encounter_trends)

encounter_year,encounter_month,encounter_year_month,total_encounters,unique_patients,total_monthly_claim_cost,total_monthly_payer_coverage,total_monthly_patient_out_of_pocket,average_claim_cost_per_encounter,average_encounter_duration_hours
1912,9,1912-09,1,1,77.49,16.62,60.87,77.49,0.25
1912,10,1912-10,1,1,77.49,16.62,60.87,77.49,0.25
1914,7,1914-07,2,1,154.98,0.00,154.98,77.49,0.25
1917,1,1917-01,2,1,154.98,0.00,154.98,77.49,0.25
1918,1,1918-01,1,1,77.49,0.00,77.49,77.49,0.25
1918,2,1918-02,1,1,77.49,0.00,77.49,77.49,0.25
1919,1,1919-01,1,1,77.49,0.00,77.49,77.49,0.25
1919,6,1919-06,1,1,15856.02,0.00,15856.02,15856.02,2.03
1921,2,1921-02,1,1,6882.62,0.00,6882.62,6882.62,2.86
1921,6,1921-06,1,1,129.16,0.00,129.16,129.16,1.0


In [0]:
from pyspark.sql import functions as F

# Monthly encounter total reconciliation
monthly_total = monthly_encounter_trends.agg(
    F.sum("total_encounters").alias("monthly_total")
).first()["monthly_total"]

gold_total = spark.table(
    "workspace.default.gold_fact_encounters"
).count()

encounter_count_check = monthly_total == gold_total

# Monthly uniqueness check
duplicate_months = (
    monthly_encounter_trends
    .groupBy("encounter_year_month")
    .count()
    .filter(F.col("count") > 1)
    .count()
)

month_uniqueness_check = duplicate_months == 0

# Validation results
print("Monthly Encounter Reconciliation:",
      "PASS" if encounter_count_check else "FAIL")

print("Monthly Key Uniqueness:",
      "PASS" if month_uniqueness_check else "FAIL")

if not encounter_count_check or not month_uniqueness_check:
    raise Exception("Monthly Encounter Trends validation failed")

print("\nMONTHLY ENCOUNTER TRENDS VALIDATION: PASS")

Monthly Encounter Reconciliation: PASS
Monthly Key Uniqueness: PASS

MONTHLY ENCOUNTER TRENDS VALIDATION: PASS


In [0]:
'''Metrics:
- Total encounters
- Total claim expenditure
- Average claim cost
- Median claim cost
- Minimum and maximum claim cost
- Total insurance coverage
- Total patient out-of-pocket estimate
- Payer coverage percentage
- Patient responsibility percentage'''

healthcare_cost_analysis = spark.sql("""
SELECT
    encounter_class,

    COUNT(encounter_id) AS total_encounters,

    ROUND(SUM(total_claim_cost), 2) AS total_claim_expenditure,

    ROUND(AVG(total_claim_cost), 2) AS average_claim_cost,

    ROUND(
        PERCENTILE_APPROX(total_claim_cost, 0.50),
        2
    ) AS median_claim_cost,

    ROUND(MIN(total_claim_cost), 2) AS minimum_claim_cost,

    ROUND(MAX(total_claim_cost), 2) AS maximum_claim_cost,

    ROUND(SUM(payer_coverage), 2) AS total_insurance_coverage,

    ROUND(SUM(patient_out_of_pocket_estimate), 2)
        AS total_patient_responsibility,

    ROUND(
        SUM(payer_coverage) / NULLIF(SUM(total_claim_cost), 0) * 100,
        2
    ) AS payer_coverage_percentage,

    ROUND(
        SUM(patient_out_of_pocket_estimate) /
        NULLIF(SUM(total_claim_cost), 0) * 100,
        2
    ) AS patient_responsibility_percentage

FROM workspace.default.gold_fact_encounters

GROUP BY encounter_class

ORDER BY total_claim_expenditure DESC
""")

display(healthcare_cost_analysis)

encounter_class,total_encounters,total_claim_expenditure,average_claim_cost,median_claim_cost,minimum_claim_cost,maximum_claim_cost,total_insurance_coverage,total_patient_responsibility,payer_coverage_percentage,patient_responsibility_percentage
ambulatory,20124,131292859.61,6524.19,645.81,0.00,83692.64,31029232.16,100263627.45,23.63,76.37
wellness,24038,45900358.66,1909.49,926.85,0.00,112296.69,14712073.09,31188285.57,32.05,67.95
outpatient,10837,30641731.90,2827.51,129.16,0.00,51733.87,7819562.34,22822169.56,25.52,74.48
emergency,2168,17184454.33,7926.41,645.81,0.00,873646.19,4034227.71,13150226.62,23.48,76.52
inpatient,1728,15147647.00,8766.00,6473.51,0.00,284854.76,2254736.54,12892910.46,14.89,85.11
urgentcare,2564,14866776.58,5798.27,129.16,129.16,53540.80,3680926.58,11185850.00,24.76,75.24


In [0]:
# Reconcile KPI #3 against the Gold fact table and KPI #1

cost_analysis_total = healthcare_cost_analysis.agg(
    {"total_claim_expenditure": "sum"}
).first()[0]

gold_claim_total = spark.sql("""
    SELECT ROUND(SUM(total_claim_cost), 2) AS total_claim_cost
    FROM workspace.default.gold_fact_encounters
""").first()["total_claim_cost"]

print("KPI #3 total expenditure:", cost_analysis_total)
print("Gold fact total claim cost:", gold_claim_total)

print("Claim expenditure reconciliation:",
      "PASS" if round(cost_analysis_total, 2) == round(gold_claim_total, 2) else "FAIL")

KPI #3 total expenditure: 255033828.08
Gold fact total claim cost: 255033828.08
Claim expenditure reconciliation: PASS


In [0]:
'''This query will calculate:
- Total encounters per payer
- Unique patients covered
- Total claim expenditure
- Total payer coverage
- Patient out-of-pocket estimate
- Average claim cost
- Average coverage per encounter
- Payer coverage percentage
- Patient responsibility percentage
- Payer ranking by total expenditure'''

payer_financial_analysis = spark.sql("""
SELECT
    payer_id,
    total_encounters,
    unique_patients,
    total_claim_cost,
    total_payer_coverage,
    total_out_of_pocket_estimate,
    average_claim_cost,

    ROUND(
        total_payer_coverage / NULLIF(total_claim_cost, 0) * 100,
        2
    ) AS payer_coverage_percentage,

    ROUND(
        total_out_of_pocket_estimate / NULLIF(total_claim_cost, 0) * 100,
        2
    ) AS patient_responsibility_percentage,

    ROUND(
        total_payer_coverage / NULLIF(total_encounters, 0),
        2
    ) AS average_coverage_per_encounter

FROM workspace.default.gold_payer_summary

ORDER BY total_claim_cost DESC
""")

display(payer_financial_analysis)

payer_id,total_encounters,unique_patients,total_claim_cost,total_payer_coverage,total_out_of_pocket_estimate,average_claim_cost,payer_coverage_percentage,patient_responsibility_percentage,average_coverage_per_encounter
b1c428d6-4f07-31e0-90f0-68ffa6ff8c76,13620,538,101549539.78,0.00,101549539.78,7455.91,0.00,100.00,0.00
7caa7254-5050-3b5e-9eae-bd5ea30e809c,8482,193,32339927.01,25643264.46,6696662.55,3812.77,79.29,20.71,3023.26
7c4411ce-02f1-39b5-b9ec-dfbea9ad3c1a,5283,363,25966027.19,24423140.77,1542886.42,4915.02,94.06,5.94,4622.97
6e2f1a2d-27bd-3701-8d08-dae202c58632,5950,887,17818677.45,11570960.86,6247716.59,2994.74,64.94,35.06,1944.70
42c4fca7-f8a9-3cd1-982a-dd9751bf3e2a,5532,862,16344629.10,2402.73,16342226.37,2954.56,0.01,99.99,0.43
4d71f845-a6a9-3c39-b242-14d25ef86a8d,5767,885,16016476.31,8466.79,16008009.52,2777.26,0.05,99.95,1.47
047f6ec3-6215-35eb-9608-f9dda363a44c,5100,861,14699780.74,5666.34,14694114.40,2882.31,0.04,99.96,1.11
d47b3510-2895-3b70-9897-342d681c769d,5635,864,14418228.81,2355.54,14415873.27,2558.69,0.02,99.98,0.42
5059a55e-5d6e-34d1-b6cb-d83d16e57bcf,5247,862,13851239.73,4643.45,13846596.28,2639.84,0.03,99.97,0.88
b3221cfc-24fb-339e-823d-bc4136cbc4ed,843,20,2029301.96,1869857.48,159444.48,2407.24,92.14,7.86,2218.10


In [0]:
from pyspark.sql import functions as F

payer_summary = spark.table(
    "workspace.default.gold_payer_summary"
)

fact_encounters = spark.table(
    "workspace.default.gold_fact_encounters"
)

# 1. Payer count reconciliation
summary_payer_count = payer_summary.count()

fact_payer_count = (
    fact_encounters
    .select("payer_id")
    .distinct()
    .count()
)

payer_count_check = summary_payer_count == fact_payer_count

# 2. Encounter count reconciliation
summary_encounter_count = payer_summary.agg(
    F.sum("total_encounters")
).first()[0]

fact_encounter_count = fact_encounters.count()

encounter_count_check = (
    summary_encounter_count == fact_encounter_count
)

# 3. Claim expenditure reconciliation
summary_claim_cost = payer_summary.agg(
    F.sum("total_claim_cost")
).first()[0]

fact_claim_cost = fact_encounters.agg(
    F.sum("total_claim_cost")
).first()[0]

claim_cost_check = (
    round(float(summary_claim_cost), 2)
    == round(float(fact_claim_cost), 2)
)

# 4. Payer coverage reconciliation
summary_coverage = payer_summary.agg(
    F.sum("total_payer_coverage")
).first()[0]

fact_coverage = fact_encounters.agg(
    F.sum("payer_coverage")
).first()[0]

coverage_check = (
    round(float(summary_coverage), 2)
    == round(float(fact_coverage), 2)
)

# 5. Patient responsibility reconciliation
summary_patient_responsibility = payer_summary.agg(
    F.sum("total_out_of_pocket_estimate")
).first()[0]

fact_patient_responsibility = fact_encounters.agg(
    F.sum("patient_out_of_pocket_estimate")
).first()[0]

patient_responsibility_check = (
    round(float(summary_patient_responsibility), 2)
    == round(float(fact_patient_responsibility), 2)
)

# 6. Payer key integrity
null_payer_keys = payer_summary.filter(
    F.col("payer_id").isNull()
).count()

duplicate_payer_keys = (
    payer_summary
    .groupBy("payer_id")
    .count()
    .filter(F.col("count") > 1)
    .count()
)

payer_key_check = (
    null_payer_keys == 0
    and duplicate_payer_keys == 0
)

# Display validation results
print("Payer Count Reconciliation:",
      "PASS" if payer_count_check else "FAIL")

print("Encounter Count Reconciliation:",
      "PASS" if encounter_count_check else "FAIL")

print("Claim Expenditure Reconciliation:",
      "PASS" if claim_cost_check else "FAIL")

print("Payer Coverage Reconciliation:",
      "PASS" if coverage_check else "FAIL")

print("Patient Responsibility Reconciliation:",
      "PASS" if patient_responsibility_check else "FAIL")

print("Payer Primary Key Integrity:",
      "PASS" if payer_key_check else "FAIL")

# Final validation
all_checks_passed = all([
    payer_count_check,
    encounter_count_check,
    claim_cost_check,
    coverage_check,
    patient_responsibility_check,
    payer_key_check
])

if not all_checks_passed:
    raise Exception("KPI #4 Payer Financial Analysis Validation Failed")

print("\nKPI #4 — PAYER FINANCIAL ANALYSIS VALIDATION: PASS")

Payer Count Reconciliation: PASS
Encounter Count Reconciliation: PASS
Claim Expenditure Reconciliation: PASS
Payer Coverage Reconciliation: PASS
Patient Responsibility Reconciliation: PASS
Payer Primary Key Integrity: PASS

KPI #4 — PAYER FINANCIAL ANALYSIS VALIDATION: PASS


In [0]:
'''We want to answer:
- Which providers handle the most encounters?
- How many unique patients does each provider serve?
- Which providers generate the highest claim expenditure?
- What is the average claim cost per provider encounter?
- How is workload distributed across providers?'''

provider_workload_analysis = spark.sql("""
SELECT
    provider_id,
    total_encounters,
    unique_patients_treated,
    distinct_encounter_classes,
    total_base_encounter_cost,
    total_claim_cost,
    total_payer_coverage,
    average_claim_cost,

    ROUND(
        total_claim_cost / NULLIF(total_encounters, 0),
        2
    ) AS recalculated_avg_claim_per_encounter,

    ROUND(
        total_payer_coverage / NULLIF(total_claim_cost, 0) * 100,
        2
    ) AS payer_coverage_percentage,

    ROUND(
        total_claim_cost / SUM(total_claim_cost) OVER () * 100,
        2
    ) AS percentage_of_overall_claim_cost

FROM workspace.default.gold_provider_summary

ORDER BY total_encounters DESC
""")

display(provider_workload_analysis)

provider_id,total_encounters,unique_patients_treated,distinct_encounter_classes,total_base_encounter_cost,total_claim_cost,total_payer_coverage,average_claim_cost,recalculated_avg_claim_per_encounter,payer_coverage_percentage,percentage_of_overall_claim_cost
43b6d00b-7643-36f8-8043-456d4eda39eb,2184,21,4,187839.36,6599341.65,981661.00,3021.68,3021.68,14.88,2.59
b0c5743a-4f9a-33b3-bab4-6a474fcf9ffd,1477,48,4,149588.33,7717062.34,1787659.63,5224.82,5224.82,23.17,3.03
faa87618-81eb-3ac8-923b-bf203be11e67,1105,56,4,113889.94,6299498.08,1867367.36,5700.90,5700.90,29.64,2.47
0bab5325-491c-326f-b1af-c28c8e4559f6,1028,33,4,107303.17,5240393.64,1031850.51,5097.66,5097.66,19.69,2.05
3aade15e-5333-3c4e-86c5-aab3511bc8e8,1020,72,4,111178.54,6168896.27,1701603.84,6047.94,6047.94,27.58,2.42
581cc2bd-9005-3008-8abb-299f40d36ca5,1010,66,4,108388.51,7111881.18,1903427.47,7041.47,7041.47,26.76,2.79
c4d2ed7d-e1b3-37e2-82f9-b21046309ba4,1000,27,4,102033.25,5177101.78,1124202.81,5177.10,5177.10,21.71,2.03
f2ee2f6a-476a-3603-9848-b5773334fc00,962,36,4,96608.47,2642916.63,167289.37,2747.31,2747.31,6.33,1.04
584a32a8-2fc6-3c4f-947a-6b2c80851a1d,941,38,4,97202.99,5073541.12,1783680.40,5391.65,5391.65,35.16,1.99
fc897c25-b2ee-30c5-a945-d0eb13a1f96e,877,39,4,89711.80,5616714.69,1152054.92,6404.46,6404.46,20.51,2.20


In [0]:
from pyspark.sql import functions as F

provider_summary = spark.table(
    "workspace.default.gold_provider_summary"
)

fact_encounters = spark.table(
    "workspace.default.gold_fact_encounters"
)

# 1. Provider count
summary_provider_count = provider_summary.count()

fact_provider_count = (
    fact_encounters
    .select("provider_id")
    .distinct()
    .count()
)

provider_count_check = (
    summary_provider_count == fact_provider_count
)

# 2. Encounter count
summary_encounter_count = provider_summary.agg(
    F.sum("total_encounters")
).first()[0]

fact_encounter_count = fact_encounters.count()

encounter_count_check = (
    summary_encounter_count == fact_encounter_count
)

# 3. Total claim cost
summary_claim_cost = provider_summary.agg(
    F.sum("total_claim_cost")
).first()[0]

fact_claim_cost = fact_encounters.agg(
    F.sum("total_claim_cost")
).first()[0]

claim_cost_check = (
    round(float(summary_claim_cost), 2)
    == round(float(fact_claim_cost), 2)
)

# 4. Total payer coverage
summary_coverage = provider_summary.agg(
    F.sum("total_payer_coverage")
).first()[0]

fact_coverage = fact_encounters.agg(
    F.sum("payer_coverage")
).first()[0]

coverage_check = (
    round(float(summary_coverage), 2)
    == round(float(fact_coverage), 2)
)

# 5. Total base encounter cost
summary_base_cost = provider_summary.agg(
    F.sum("total_base_encounter_cost")
).first()[0]

fact_base_cost = fact_encounters.agg(
    F.sum("base_encounter_cost")
).first()[0]

base_cost_check = (
    round(float(summary_base_cost), 2)
    == round(float(fact_base_cost), 2)
)

# 6. Provider primary key integrity
null_provider_keys = provider_summary.filter(
    F.col("provider_id").isNull()
).count()

duplicate_provider_keys = (
    provider_summary
    .groupBy("provider_id")
    .count()
    .filter(F.col("count") > 1)
    .count()
)

provider_key_check = (
    null_provider_keys == 0
    and duplicate_provider_keys == 0
)

# Display checks
print("Provider Count Reconciliation:",
      "PASS" if provider_count_check else "FAIL")

print("Encounter Count Reconciliation:",
      "PASS" if encounter_count_check else "FAIL")

print("Claim Expenditure Reconciliation:",
      "PASS" if claim_cost_check else "FAIL")

print("Payer Coverage Reconciliation:",
      "PASS" if coverage_check else "FAIL")

print("Base Encounter Cost Reconciliation:",
      "PASS" if base_cost_check else "FAIL")

print("Provider Primary Key Integrity:",
      "PASS" if provider_key_check else "FAIL")

# Final validation
all_checks_passed = all([
    provider_count_check,
    encounter_count_check,
    claim_cost_check,
    coverage_check,
    base_cost_check,
    provider_key_check
])

if not all_checks_passed:
    raise Exception("KPI #5 Provider Workload Validation Failed")

print("\nKPI #5 — PROVIDER WORKLOAD VALIDATION: PASS")

Provider Count Reconciliation: PASS
Encounter Count Reconciliation: PASS
Claim Expenditure Reconciliation: PASS
Payer Coverage Reconciliation: PASS
Base Encounter Cost Reconciliation: PASS
Provider Primary Key Integrity: PASS

KPI #5 — PROVIDER WORKLOAD VALIDATION: PASS


In [0]:
'''We will answer:
- Which organizations handle the most encounters?
- How many unique patients does each organization serve?
- How many providers operate across each organization?
- Which organizations generate the highest claims expenditure?
- What is the average claim cost per encounter?
- How is utilization distributed across organizations?'''
organization_performance_analysis = spark.sql("""
SELECT
    organization_id,
    total_encounters,
    unique_patients_served,
    unique_providers,
    distinct_encounter_classes,
    total_base_encounter_cost,
    total_claim_cost,
    total_payer_coverage,
    average_claim_cost,

    ROUND(
        total_claim_cost / NULLIF(total_encounters, 0),
        2
    ) AS recalculated_avg_claim_per_encounter,

    ROUND(
        total_payer_coverage / NULLIF(total_claim_cost, 0) * 100,
        2
    ) AS payer_coverage_percentage,

    ROUND(
        total_claim_cost / SUM(total_claim_cost) OVER () * 100,
        2
    ) AS percentage_of_overall_claim_cost

FROM workspace.default.gold_organization_summary

ORDER BY total_encounters DESC
""")

display(organization_performance_analysis)

organization_id,total_encounters,unique_patients_served,unique_providers,distinct_encounter_classes,total_base_encounter_cost,total_claim_cost,total_payer_coverage,average_claim_cost,recalculated_avg_claim_per_encounter,payer_coverage_percentage,percentage_of_overall_claim_cost
6f122869-a856-3d65-8db9-099bf4f5bbb8,2184,21,1,4,187839.36,6599341.65,981661.00,3021.68,3021.68,14.88,2.59
4861d01f-019c-3dac-a153-8334e50919f9,1477,48,1,4,149588.33,7717062.34,1787659.63,5224.82,5224.82,23.17,3.03
d692e283-0833-3201-8e55-4f868a9c0736,1105,56,1,4,113889.94,6299498.08,1867367.36,5700.90,5700.90,29.64,2.47
b1ddf812-1fdd-3adf-b1d5-32cc8bd07ebb,1028,33,1,4,107303.17,5240393.64,1031850.51,5097.66,5097.66,19.69,2.05
69176529-fd1f-3b3f-abce-a0a3626769eb,1020,72,1,4,111178.54,6168896.27,1701603.84,6047.94,6047.94,27.58,2.42
e002090d-4e92-300e-b41e-7d1f21dee4c6,1010,66,1,4,108388.51,7111881.18,1903427.47,7041.47,7041.47,26.76,2.79
8ad64ecf-c817-3753-bee7-006a8e662e06,1000,27,1,4,102033.25,5177101.78,1124202.81,5177.10,5177.10,21.71,2.03
23834663-ed53-3da9-b330-d6e1ecb8428e,962,36,1,4,96608.47,2642916.63,167289.37,2747.31,2747.31,6.33,1.04
b0e04623-b02c-3f8b-92ea-943fc4db60da,941,38,1,4,97202.99,5073541.12,1783680.40,5391.65,5391.65,35.16,1.99
d5117822-5756-389d-9547-891a372d580f,877,39,1,4,89711.80,5616714.69,1152054.92,6404.46,6404.46,20.51,2.20


In [0]:
from pyspark.sql import functions as F

organization_summary = spark.table(
    "workspace.default.gold_organization_summary"
)

fact_encounters = spark.table(
    "workspace.default.gold_fact_encounters"
)

# 1. Organization count reconciliation
summary_organization_count = organization_summary.count()

fact_organization_count = (
    fact_encounters
    .select("organization_id")
    .distinct()
    .count()
)

organization_count_check = (
    summary_organization_count == fact_organization_count
)

# 2. Encounter count reconciliation
summary_encounter_count = organization_summary.agg(
    F.sum("total_encounters")
).first()[0]

fact_encounter_count = fact_encounters.count()

encounter_count_check = (
    summary_encounter_count == fact_encounter_count
)

# 3. Claim expenditure reconciliation
summary_claim_cost = organization_summary.agg(
    F.sum("total_claim_cost")
).first()[0]

fact_claim_cost = fact_encounters.agg(
    F.sum("total_claim_cost")
).first()[0]

claim_cost_check = (
    round(float(summary_claim_cost), 2)
    == round(float(fact_claim_cost), 2)
)

# 4. Payer coverage reconciliation
summary_coverage = organization_summary.agg(
    F.sum("total_payer_coverage")
).first()[0]

fact_coverage = fact_encounters.agg(
    F.sum("payer_coverage")
).first()[0]

coverage_check = (
    round(float(summary_coverage), 2)
    == round(float(fact_coverage), 2)
)

# 5. Base encounter cost reconciliation
summary_base_cost = organization_summary.agg(
    F.sum("total_base_encounter_cost")
).first()[0]

fact_base_cost = fact_encounters.agg(
    F.sum("base_encounter_cost")
).first()[0]

base_cost_check = (
    round(float(summary_base_cost), 2)
    == round(float(fact_base_cost), 2)
)

# 6. Organization primary key integrity
null_organization_keys = organization_summary.filter(
    F.col("organization_id").isNull()
).count()

duplicate_organization_keys = (
    organization_summary
    .groupBy("organization_id")
    .count()
    .filter(F.col("count") > 1)
    .count()
)

organization_key_check = (
    null_organization_keys == 0
    and duplicate_organization_keys == 0
)

# Display results
print("Organization Count Reconciliation:",
      "PASS" if organization_count_check else "FAIL")

print("Encounter Count Reconciliation:",
      "PASS" if encounter_count_check else "FAIL")

print("Claim Expenditure Reconciliation:",
      "PASS" if claim_cost_check else "FAIL")

print("Payer Coverage Reconciliation:",
      "PASS" if coverage_check else "FAIL")

print("Base Encounter Cost Reconciliation:",
      "PASS" if base_cost_check else "FAIL")

print("Organization Primary Key Integrity:",
      "PASS" if organization_key_check else "FAIL")

# Final validation
all_checks_passed = all([
    organization_count_check,
    encounter_count_check,
    claim_cost_check,
    coverage_check,
    base_cost_check,
    organization_key_check
])

if not all_checks_passed:
    raise Exception("KPI #6 Organization Utilization Validation Failed")

print("\nKPI #6 — ORGANIZATION UTILIZATION VALIDATION: PASS")

Organization Count Reconciliation: PASS
Encounter Count Reconciliation: PASS
Claim Expenditure Reconciliation: PASS
Payer Coverage Reconciliation: PASS
Base Encounter Cost Reconciliation: PASS
Organization Primary Key Integrity: PASS

KPI #6 — ORGANIZATION UTILIZATION VALIDATION: PASS


In [0]:
'''We will analyze:
- Most frequently recorded conditions
- Unique patients affected by each condition
- Total condition records
- Conditions with ongoing records vs. records with an end date
- Average condition duration
- Ranking of conditions by patient prevalence'''

condition_prevalence_analysis = spark.sql("""
SELECT
    condition_code,
    condition_description,
    total_condition_records,
    unique_patients_affected,

    ROUND(
        unique_patients_affected / 1163.0 * 100,
        2
    ) AS percentage_of_total_patients,

    records_with_end_date,
    records_without_end_date,
    average_condition_duration_days,
    first_recorded_date,
    last_recorded_date

FROM workspace.default.gold_condition_summary

ORDER BY unique_patients_affected DESC,
         total_condition_records DESC
""")

display(condition_prevalence_analysis)

condition_code,condition_description,total_condition_records,unique_patients_affected,percentage_of_total_patients,records_with_end_date,records_without_end_date,average_condition_duration_days,first_recorded_date,last_recorded_date
73595000,Stress (finding),5137,894,76.87,4613,524,902.18,1929-12-28,2021-11-09
160903007,Full-time employment (finding),13805,834,71.71,13209,596,405.2,1929-09-06,2021-11-11
444814009,Viral sinusitis (disorder),1233,739,63.54,1222,11,16.84,1922-08-13,2021-11-06
160904001,Part-time employment (finding),2426,686,58.99,2270,156,363.85,1931-12-07,2021-11-09
423315002,Limited social contact (finding),1200,661,56.84,1033,167,2516.57,1929-12-28,2021-11-05
422650009,Social isolation (finding),1243,653,56.15,1083,160,2321.52,1931-08-04,2021-11-07
224299000,Received higher education (finding),523,523,44.97,0,523,null,1929-09-06,2021-11-02
195662009,Acute viral pharyngitis (disorder),678,506,43.51,675,3,10.55,1937-12-28,2021-11-10
741062008,Not in labor force (finding),1077,501,43.08,993,84,366.91,1932-08-09,2021-11-09
706893006,Victim of intimate partner abuse (finding),819,466,40.07,622,197,1050.36,1931-08-04,2021-11-08


In [0]:
from pyspark.sql import functions as F

condition_summary = spark.table(
    "workspace.default.gold_condition_summary"
)

silver_conditions = spark.table(
    "workspace.default.healthcare_silver_conditions"
)

# 1. Condition record reconciliation
summary_condition_records = condition_summary.agg(
    F.sum("total_condition_records")
).first()[0]

silver_condition_records = silver_conditions.count()

condition_record_check = (
    summary_condition_records == silver_condition_records
)

# 2. Condition group count reconciliation
summary_condition_groups = condition_summary.count()

silver_condition_groups = (
    silver_conditions
    .select("condition_code", "condition_description")
    .distinct()
    .count()
)

condition_group_check = (
    summary_condition_groups == silver_condition_groups
)

# 3. Null condition keys
null_condition_keys = condition_summary.filter(
    F.col("condition_code").isNull()
    | F.col("condition_description").isNull()
).count()

null_condition_check = null_condition_keys == 0

# 4. Duplicate composite keys
duplicate_condition_keys = (
    condition_summary
    .groupBy("condition_code", "condition_description")
    .count()
    .filter(F.col("count") > 1)
    .count()
)

duplicate_condition_check = duplicate_condition_keys == 0

# Print validation results
print("Condition Record Reconciliation:",
      "PASS" if condition_record_check else "FAIL")

print("Condition Group Count Reconciliation:",
      "PASS" if condition_group_check else "FAIL")

print("Null Condition Key Check:",
      "PASS" if null_condition_check else "FAIL")

print("Condition Composite Key Integrity:",
      "PASS" if duplicate_condition_check else "FAIL")

# Final validation
all_checks_passed = all([
    condition_record_check,
    condition_group_check,
    null_condition_check,
    duplicate_condition_check
])

if not all_checks_passed:
    raise Exception("KPI #7 Condition Analysis Validation Failed")

print("\nKPI #7 — CONDITION PREVALENCE VALIDATION: PASS")

Condition Record Reconciliation: PASS
Condition Group Count Reconciliation: PASS
Null Condition Key Check: PASS
Condition Composite Key Integrity: PASS

KPI #7 — CONDITION PREVALENCE VALIDATION: PASS


In [0]:
'''Answer these business questions:
- Which procedures are performed most frequently?
- How many unique patients receive each procedure?
- Which procedures generate the highest total cost?
- Which procedures have the highest average cost?
- How much time is spent on procedures?
- Which procedures are more frequent versus more expensive?'''

procedure_utilization_analysis = spark.sql("""
SELECT
    procedure_code,
    procedure_description,

    total_procedures,
    unique_patients,

    ROUND(
        unique_patients / 1163.0 * 100,
        2
    ) AS percentage_of_total_patients,

    total_procedure_cost,
    average_procedure_cost,

    ROUND(
        total_procedure_cost / SUM(total_procedure_cost) OVER () * 100,
        2
    ) AS percentage_of_total_procedure_cost,

    total_procedure_hours,
    average_procedure_hours,

    first_procedure_datetime,
    last_procedure_datetime

FROM workspace.default.gold_procedure_summary

ORDER BY total_procedure_cost DESC
""")

display(procedure_utilization_analysis)

procedure_code,procedure_description,total_procedures,unique_patients,percentage_of_total_patients,total_procedure_cost,average_procedure_cost,percentage_of_total_procedure_cost,total_procedure_hours,average_procedure_hours,first_procedure_datetime,last_procedure_datetime
180325003,Electrical cardioversion,964,43,3.70,30001802.37,31122.20,14.27,241.0,0.25,1982-01-19T00:08:52.000Z,2021-11-18T22:59:05.000Z
180256009,Subcutaneous immunotherapy,1929,44,3.78,27532634.03,14273.01,13.10,1205.95,0.63,1990-01-23T11:58:32.000Z,2021-11-16T07:00:59.000Z
225158009,Auscultation of the fetal heart,2717,192,16.51,17287662.46,6362.78,8.22,679.25,0.25,1934-02-12T19:51:58.000Z,2021-11-17T21:04:01.000Z
274804006,Evaluation of uterine fundal height,2717,192,16.51,17179277.40,6322.88,8.17,679.25,0.25,1934-02-12T19:51:58.000Z,2021-11-17T21:04:01.000Z
703423002,Combined chemotherapy and radiation therapy (procedure),690,27,2.32,9894054.89,14339.21,4.71,1223.18,1.77,1975-06-03T22:17:15.000Z,2021-11-18T14:01:22.000Z
73761001,Colonoscopy,538,285,24.51,7697553.15,14307.72,3.66,314.21,0.58,1963-06-11T21:04:17.000Z,2021-10-31T11:45:49.000Z
169230002,Ultrasound scan for fetal viability,477,211,18.14,5409882.33,11341.47,2.57,119.25,0.25,1934-02-12T19:51:58.000Z,2021-11-15T13:24:23.000Z
710824005,Assessment of health and social care needs (procedure),8901,912,78.42,4598701.65,516.65,2.19,6678.75,0.75,1934-12-10T19:51:58.000Z,2021-11-18T10:36:25.000Z
76601001,Intramuscular injection,1450,112,9.63,4509300.89,3109.86,2.15,358.34,0.25,1994-08-28T02:56:17.000Z,2021-11-19T16:50:22.000Z
228557008,Cognitive and behavioral therapy,322,16,1.38,4051536.24,12582.41,1.93,322.0,1.0,2011-12-23T03:50:03.000Z,2021-11-03T08:57:05.000Z


In [0]:
from pyspark.sql import functions as F

procedure_summary = spark.table(
    "workspace.default.gold_procedure_summary"
)

silver_procedures = spark.table(
    "workspace.default.healthcare_silver_procedures"
)

# 1. Procedure record reconciliation
summary_procedure_records = procedure_summary.agg(
    F.sum("total_procedures")
).first()[0]

silver_procedure_records = silver_procedures.count()

procedure_record_check = (
    summary_procedure_records == silver_procedure_records
)

# 2. Procedure group count reconciliation
summary_procedure_groups = procedure_summary.count()

silver_procedure_groups = (
    silver_procedures
    .select("procedure_code", "procedure_description")
    .distinct()
    .count()
)

procedure_group_check = (
    summary_procedure_groups == silver_procedure_groups
)

# 3. Null composite key check
null_procedure_keys = procedure_summary.filter(
    F.col("procedure_code").isNull()
    | F.col("procedure_description").isNull()
).count()

null_procedure_check = null_procedure_keys == 0

# 4. Duplicate composite key check
duplicate_procedure_keys = (
    procedure_summary
    .groupBy("procedure_code", "procedure_description")
    .count()
    .filter(F.col("count") > 1)
    .count()
)

duplicate_procedure_check = duplicate_procedure_keys == 0

# Display results
print("Procedure Record Reconciliation:",
      "PASS" if procedure_record_check else "FAIL")

print("Procedure Group Count Reconciliation:",
      "PASS" if procedure_group_check else "FAIL")

print("Null Procedure Key Check:",
      "PASS" if null_procedure_check else "FAIL")

print("Procedure Composite Key Integrity:",
      "PASS" if duplicate_procedure_check else "FAIL")

# Final validation
all_checks_passed = all([
    procedure_record_check,
    procedure_group_check,
    null_procedure_check,
    duplicate_procedure_check
])

if not all_checks_passed:
    raise Exception("KPI #8 Procedure Analysis Validation Failed")

print("\nKPI #8 — PROCEDURE UTILIZATION VALIDATION: PASS")

Procedure Record Reconciliation: PASS
Procedure Group Count Reconciliation: PASS
Null Procedure Key Check: PASS
Procedure Composite Key Integrity: PASS

KPI #8 — PROCEDURE UTILIZATION VALIDATION: PASS


In [0]:
from pyspark.sql import functions as F

# ============================================================
# PHASE 7.2 — DATA QUALITY THRESHOLD MONITORING
# ============================================================

quality_thresholds = {
    "gold_fact_encounters": {
        "min_rows": 1,
        "key_columns": ["encounter_id"]
    },
    "gold_patient_summary": {
        "min_rows": 1,
        "key_columns": ["patient_id"]
    },
    "gold_condition_summary": {
        "min_rows": 1,
        "key_columns": ["condition_code", "condition_description"]
    },
    "gold_procedure_summary": {
        "min_rows": 1,
        "key_columns": ["procedure_code", "procedure_description"]
    },
    "gold_organization_summary": {
        "min_rows": 1,
        "key_columns": ["organization_id"]
    },
    "gold_provider_summary": {
        "min_rows": 1,
        "key_columns": ["provider_id"]
    },
    "gold_payer_summary": {
        "min_rows": 1,
        "key_columns": ["payer_id"]
    }
}

quality_results = []

for table_name, config in quality_thresholds.items():

    df = spark.table(f"workspace.default.{table_name}")

    total_rows = df.count()

    # Check minimum row threshold
    minimum_rows_failed = total_rows < config["min_rows"]

    # Check null business/primary keys
    null_condition = None

    for key in config["key_columns"]:
        condition = F.col(key).isNull()

        null_condition = (
            condition
            if null_condition is None
            else null_condition | condition
        )

    null_key_rows = df.filter(null_condition).count()

    # Check duplicates at the actual table grain
    duplicate_keys = (
        df.groupBy(*config["key_columns"])
          .count()
          .filter(F.col("count") > 1)
          .count()
    )

    status = (
        "FAIL"
        if minimum_rows_failed
        or null_key_rows > 0
        or duplicate_keys > 0
        else "PASS"
    )

    quality_results.append(
        (
            table_name,
            total_rows,
            config["min_rows"],
            null_key_rows,
            duplicate_keys,
            status
        )
    )

# Create monitoring report
quality_monitoring_report = spark.createDataFrame(
    quality_results,
    [
        "table_name",
        "actual_rows",
        "minimum_expected_rows",
        "null_key_rows",
        "duplicate_key_count",
        "status"
    ]
)

quality_monitoring_report.show(truncate=False)

# Stop the process if any critical Gold table fails
failed_quality_checks = (
    quality_monitoring_report
    .filter(F.col("status") == "FAIL")
    .count()
)

if failed_quality_checks > 0:
    raise Exception(
        f"DATA QUALITY FAILURE: {failed_quality_checks} Gold tables failed monitoring"
    )

print("PHASE 7.2 DATA QUALITY MONITORING: PASS")

+-------------------------+-----------+---------------------+-------------+-------------------+------+
|table_name               |actual_rows|minimum_expected_rows|null_key_rows|duplicate_key_count|status|
+-------------------------+-----------+---------------------+-------------+-------------------+------+
|gold_fact_encounters     |61459      |1                    |0            |0                  |PASS  |
|gold_patient_summary     |1163       |1                    |0            |0                  |PASS  |
|gold_condition_summary   |204        |1                    |0            |0                  |PASS  |
|gold_procedure_summary   |178        |1                    |0            |0                  |PASS  |
|gold_organization_summary|1122       |1                    |0            |0                  |PASS  |
|gold_provider_summary    |1123       |1                    |0            |0                  |PASS  |
|gold_payer_summary       |10         |1                    |0           

In [0]:
from pyspark.sql import functions as F
from pyspark.sql import types as T
import uuid

# Unique ID for this execution
run_id = str(uuid.uuid4())

# Create audit records from the existing Phase 7.2 monitoring results
audit_df = (
    quality_monitoring_report
    .withColumn("run_id", F.lit(run_id))
    .withColumn("run_timestamp", F.current_timestamp())
    .withColumn("pipeline_name", F.lit("healthcare_batch_pipeline"))
    .withColumn("audit_date", F.current_date())
    .select(
        "run_id",
        "run_timestamp",
        "audit_date",
        "pipeline_name",
        "table_name",
        "actual_rows",
        "minimum_expected_rows",
        "null_key_rows",
        "duplicate_key_count",
        "status"
    )
)

# Persistent Delta audit table
audit_table = "workspace.default.healthcare_data_quality_audit"

# Append audit history (preserves previous runs)
(
    audit_df.write
    .format("delta")
    .mode("append")
    .saveAsTable(audit_table)
)

print(f"Audit log saved successfully. Run ID: {run_id}")

Audit log saved successfully. Run ID: 5983f430-c734-466f-be3a-1de7c17578d7


In [0]:
audit_log = spark.table("workspace.default.healthcare_data_quality_audit")

# Show latest execution
audit_log.filter(F.col("run_id") == run_id) \
    .orderBy("table_name") \
    .show(truncate=False)

# Audit history summary
audit_log.groupBy("run_id", "run_timestamp", "status") \
    .agg(
        F.count("*").alias("tables_checked"),
        F.sum("actual_rows").alias("total_rows_monitored"),
        F.sum("null_key_rows").alias("total_null_keys"),
        F.sum("duplicate_key_count").alias("total_duplicate_keys")
    ) \
    .orderBy(F.col("run_timestamp").desc()) \
    .show(truncate=False)

+------------------------------------+--------------------------+----------+-------------------------+-------------------------+-----------+---------------------+-------------+-------------------+------+
|run_id                              |run_timestamp             |audit_date|pipeline_name            |table_name               |actual_rows|minimum_expected_rows|null_key_rows|duplicate_key_count|status|
+------------------------------------+--------------------------+----------+-------------------------+-------------------------+-----------+---------------------+-------------+-------------------+------+
|5983f430-c734-466f-be3a-1de7c17578d7|2026-11-18 14:43:37.870234|2026-11-18|healthcare_batch_pipeline|gold_condition_summary   |204        |1                    |0            |0                  |PASS  |
|5983f430-c734-466f-be3a-1de7c17578d7|2026-11-18 14:43:37.870234|2026-11-18|healthcare_batch_pipeline|gold_fact_encounters     |61459      |1                    |0            |0       

In [0]:
from pyspark.sql import functions as F

# Required Gold tables
required_gold_tables = [
    "gold_fact_encounters",
    "gold_patient_summary",
    "gold_condition_summary",
    "gold_procedure_summary",
    "gold_organization_summary",
    "gold_provider_summary",
    "gold_payer_summary"
]

audit_table = "workspace.default.healthcare_data_quality_audit"

recovery_results = []

# 1. Check Gold table availability
for table_name in required_gold_tables:
    full_table_name = f"workspace.default.{table_name}"

    try:
        df = spark.table(full_table_name)
        row_count = df.count()

        recovery_results.append(
            (table_name, "AVAILABLE", row_count, "PASS")
        )

    except Exception as e:
        recovery_results.append(
            (table_name, "UNAVAILABLE", 0, "FAIL")
        )

# Display Gold availability
recovery_report = spark.createDataFrame(
    recovery_results,
    ["table_name", "availability", "row_count", "status"]
)

recovery_report.show(truncate=False)

# 2. Check audit history availability
audit_exists = spark.catalog.tableExists(audit_table)

if audit_exists:
    audit_df = spark.table(audit_table)

    # Find the latest pipeline execution
    latest_run = (
        audit_df.orderBy(F.col("run_timestamp").desc())
        .select("run_id")
        .first()
    )

    if latest_run:
        latest_run_id = latest_run["run_id"]

        latest_audit = audit_df.filter(
            F.col("run_id") == latest_run_id
        )

        total_tables = latest_audit.count()
        failed_tables = latest_audit.filter(
            F.col("status") != "PASS"
        ).count()

        audit_status = (
            "PASS"
            if total_tables == 7 and failed_tables == 0
            else "FAIL"
        )

        print(f"Latest Run ID: {latest_run_id}")
        print(f"Tables Audited: {total_tables}")
        print(f"Failed Tables: {failed_tables}")
        print(f"Audit Status: {audit_status}")

    else:
        audit_status = "FAIL"
        print("No pipeline audit records found.")

else:
    audit_status = "FAIL"
    print("Audit table is unavailable.")

# 3. Final recovery readiness
gold_failures = recovery_report.filter(
    F.col("status") == "FAIL"
).count()

if gold_failures == 0 and audit_status == "PASS":
    print("FAILURE RECOVERY READINESS: PASS")
    print("Gold tables are accessible and the latest audited run is healthy.")
else:
    print("FAILURE RECOVERY READINESS: FAIL")
    print("Investigate unavailable Gold tables or failed audit records.")

+-------------------------+------------+---------+------+
|table_name               |availability|row_count|status|
+-------------------------+------------+---------+------+
|gold_fact_encounters     |AVAILABLE   |61459    |PASS  |
|gold_patient_summary     |AVAILABLE   |1163     |PASS  |
|gold_condition_summary   |AVAILABLE   |204      |PASS  |
|gold_procedure_summary   |AVAILABLE   |178      |PASS  |
|gold_organization_summary|AVAILABLE   |1122     |PASS  |
|gold_provider_summary    |AVAILABLE   |1123     |PASS  |
|gold_payer_summary       |AVAILABLE   |10       |PASS  |
+-------------------------+------------+---------+------+

Latest Run ID: 5983f430-c734-466f-be3a-1de7c17578d7
Tables Audited: 7
Failed Tables: 0
Audit Status: PASS
FAILURE RECOVERY READINESS: PASS
Gold tables are accessible and the latest audited run is healthy.


In [0]:
from pyspark.sql import functions as F

# Expected Gold table counts from the validated pipeline
expected_gold_counts = {
    "gold_fact_encounters": 61459,
    "gold_patient_summary": 1163,
    "gold_condition_summary": 204,
    "gold_procedure_summary": 178,
    "gold_organization_summary": 1122,
    "gold_provider_summary": 1123,
    "gold_payer_summary": 10
}

# Correct keys for each Gold table
gold_primary_keys = {
    "gold_fact_encounters": ["encounter_id"],
    "gold_patient_summary": ["patient_id"],
    "gold_condition_summary": ["condition_code", "condition_description"],
    "gold_procedure_summary": ["procedure_code", "procedure_description"],
    "gold_organization_summary": ["organization_id"],
    "gold_provider_summary": ["provider_id"],
    "gold_payer_summary": ["payer_id"]
}

rerun_results = []

for table_name, expected_count in expected_gold_counts.items():

    full_table_name = f"workspace.default.{table_name}"
    df = spark.table(full_table_name)

    actual_count = df.count()

    # Count duplicate records at the correct table grain
    key_columns = gold_primary_keys[table_name]

    duplicate_count = (
        df.groupBy(*key_columns)
        .count()
        .filter(F.col("count") > 1)
        .count()
    )

    # Validate count stability and uniqueness
    count_check = actual_count == expected_count
    uniqueness_check = duplicate_count == 0

    status = (
        "PASS"
        if count_check and uniqueness_check
        else "FAIL"
    )

    rerun_results.append(
        (
            table_name,
            expected_count,
            actual_count,
            duplicate_count,
            count_check,
            uniqueness_check,
            status
        )
    )

# Build validation report
safe_rerun_report = spark.createDataFrame(
    rerun_results,
    [
        "table_name",
        "expected_rows",
        "actual_rows",
        "duplicate_key_count",
        "row_count_stable",
        "keys_unique",
        "status"
    ]
)

safe_rerun_report.show(truncate=False)

# Overall validation
failed_tables = safe_rerun_report.filter(
    F.col("status") == "FAIL"
).count()

if failed_tables == 0:
    print("PHASE 7.5 IDEMPOTENCY VALIDATION: PASS")
    print("Gold row counts are stable and keys remain unique.")
    print("Safe rerun validation completed without modifying any data.")
else:
    print(f"PHASE 7.5 VALIDATION: FAIL — {failed_tables} tables require investigation.")

+-------------------------+-------------+-----------+-------------------+----------------+-----------+------+
|table_name               |expected_rows|actual_rows|duplicate_key_count|row_count_stable|keys_unique|status|
+-------------------------+-------------+-----------+-------------------+----------------+-----------+------+
|gold_fact_encounters     |61459        |61459      |0                  |true            |true       |PASS  |
|gold_patient_summary     |1163         |1163       |0                  |true            |true       |PASS  |
|gold_condition_summary   |204          |204        |0                  |true            |true       |PASS  |
|gold_procedure_summary   |178          |178        |0                  |true            |true       |PASS  |
|gold_organization_summary|1122         |1122       |0                  |true            |true       |PASS  |
|gold_provider_summary    |1123         |1123       |0                  |true            |true       |PASS  |
|gold_paye

# Healthcare Batch Pipeline — Operational Runbook

## 1. Pipeline Overview
**Pipeline name:** healthcare_batch_pipeline  
**Platform:** Databricks Free Edition (Serverless)  
**Architecture:** Bronze → Silver → Gold  
**Processing type:** Scheduled batch processing  
**Schedule:** Daily at 6:00 AM IST

## 2. Pipeline Monitoring
Monitor the Databricks Job page for:
- Job execution status (Succeeded / Failed)
- Execution duration
- Scheduled run status
- Error logs and failed task details
- Notification delivery

## 3. Data Quality Monitoring
Execute the Phase 7.2 monitoring cell.

Expected conditions:
- All seven Gold tables are accessible.
- No null primary/composite keys.
- No duplicate keys at the defined table grain.
- Gold tables are populated.

Expected result: DATA QUALITY MONITORING — PASS.

## 4. Audit Log Investigation
Audit table:

`workspace.default.healthcare_data_quality_audit`

Use the audit log to review:
- Run ID
- Execution timestamp
- Table-level row counts
- Null-key counts
- Duplicate-key counts
- Validation status

## 5. Failure Troubleshooting

| Failure | Investigation | Action |
|---|---|---|
| Job failed | Inspect job error and failed task | Correct the root cause |
| Gold table missing | Check upstream processing and table availability | Restore the pipeline output |
| Unexpected row count | Compare against previous run and source counts | Investigate missing or duplicate records |
| Null keys | Inspect Silver source and transformation logic | Correct invalid key records |
| Duplicate keys | Check aggregation and joins | Fix grain or join logic |
| Audit log missing | Check audit logging cell and table existence | Restore logging execution |

## 6. Recovery Procedure
1. Identify the failed pipeline stage.
2. Review the Databricks execution logs.
3. Determine the root cause.
4. Validate the upstream data.
5. Correct the issue.
6. Rerun only when the pipeline is safe to execute.
7. Execute data quality monitoring.
8. Verify the audit log.
9. Confirm all Gold tables are healthy.

## 7. Safe Rerun Checklist
- [ ] Root cause identified and fixed.
- [ ] Source data is available.
- [ ] No unintended partial writes.
- [ ] Pipeline write operations support safe reruns.
- [ ] Gold keys are unique.
- [ ] Row counts are reconciled.
- [ ] Data quality validation passes.
- [ ] Audit log reflects the latest run.

## 8. Operational Success Criteria
The pipeline is considered healthy when:
- Job execution succeeds.
- All seven Gold tables are available.
- Table-level data quality checks pass.
- No unexpected duplicate or null keys exist.
- Row counts are reconciled.
- Audit records are successfully generated.

**Owner:** Data Engineering  
**Purpose:** Maintain pipeline reliability, data integrity, and operational recoverability.

In [0]:
from pyspark.sql import functions as F

# ============================================================
# PHASE 7.7 - FINAL PIPELINE RELIABILITY REVIEW
# Read-only validation of the Healthcare Batch Pipeline
# ============================================================

expected_gold = {
    "gold_fact_encounters": {
        "expected_rows": 61459,
        "keys": ["encounter_id"]
    },
    "gold_patient_summary": {
        "expected_rows": 1163,
        "keys": ["patient_id"]
    },
    "gold_condition_summary": {
        "expected_rows": 204,
        "keys": ["condition_code", "condition_description"]
    },
    "gold_procedure_summary": {
        "expected_rows": 178,
        "keys": ["procedure_code", "procedure_description"]
    },
    "gold_organization_summary": {
        "expected_rows": 1122,
        "keys": ["organization_id"]
    },
    "gold_provider_summary": {
        "expected_rows": 1123,
        "keys": ["provider_id"]
    },
    "gold_payer_summary": {
        "expected_rows": 10,
        "keys": ["payer_id"]
    }
}

reliability_results = []

# ------------------------------------------------------------
# 1. Gold Table Availability, Volume and Key Integrity
# ------------------------------------------------------------

for table_name, config in expected_gold.items():

    full_name = f"workspace.default.{table_name}"

    try:
        df = spark.table(full_name)

        actual_rows = df.count()
        expected_rows = config["expected_rows"]
        key_columns = config["keys"]

        null_condition = None

        for key in key_columns:
            condition = F.col(key).isNull()
            null_condition = (
                condition if null_condition is None
                else null_condition | condition
            )

        null_keys = df.filter(null_condition).count()

        duplicate_keys = (
            df.groupBy(*key_columns)
              .count()
              .filter(F.col("count") > 1)
              .count()
        )

        count_pass = actual_rows == expected_rows
        null_pass = null_keys == 0
        duplicate_pass = duplicate_keys == 0

        overall_status = (
            "PASS"
            if count_pass and null_pass and duplicate_pass
            else "FAIL"
        )

        reliability_results.append((
            table_name,
            actual_rows,
            expected_rows,
            null_keys,
            duplicate_keys,
            overall_status
        ))

    except Exception as e:

        reliability_results.append((
            table_name,
            -1,
            config["expected_rows"],
            -1,
            -1,
            "FAIL"
        ))

# ------------------------------------------------------------
# 2. Display Gold Reliability Report
# ------------------------------------------------------------

reliability_report = spark.createDataFrame(
    reliability_results,
    [
        "table_name",
        "actual_rows",
        "expected_rows",
        "null_key_count",
        "duplicate_key_count",
        "status"
    ]
)

print("GOLD TABLE RELIABILITY REPORT")
reliability_report.show(truncate=False)

gold_failures = reliability_report.filter(
    F.col("status") == "FAIL"
).count()

# ------------------------------------------------------------
# 3. Audit Table Availability and Latest Run Validation
# ------------------------------------------------------------

audit_table = "workspace.default.healthcare_data_quality_audit"

audit_status = "FAIL"
audit_tables_checked = 0
audit_failed_tables = -1

if spark.catalog.tableExists(audit_table):

    audit_df = spark.table(audit_table)

    latest_run_row = (
        audit_df.orderBy(F.col("run_timestamp").desc())
        .select("run_id")
        .first()
    )

    if latest_run_row is not None:

        latest_run_id = latest_run_row["run_id"]

        latest_audit = audit_df.filter(
            F.col("run_id") == latest_run_id
        )

        audit_tables_checked = latest_audit.count()

        audit_failed_tables = latest_audit.filter(
            F.col("status") != "PASS"
        ).count()

        audit_status = (
            "PASS"
            if audit_tables_checked == 7
            and audit_failed_tables == 0
            else "FAIL"
        )

        print("\nLATEST AUDIT RUN")
        print(f"Run ID: {latest_run_id}")
        print(f"Tables checked: {audit_tables_checked}")
        print(f"Failed audit tables: {audit_failed_tables}")
        print(f"Audit status: {audit_status}")

    else:
        print("Audit table exists but contains no execution records.")

else:
    print("Audit table is missing.")

# ------------------------------------------------------------
# 4. Final Reliability Decision
# ------------------------------------------------------------

print("\n" + "=" * 60)
print("FINAL HEALTH SUMMARY")
print("=" * 60)

print(f"Gold table failures       : {gold_failures}")
print(f"Audit validation         : {audit_status}")
print(f"Audit tables checked     : {audit_tables_checked}")

if gold_failures == 0 and audit_status == "PASS":

    print("\nHEALTHCARE BATCH PIPELINE RELIABILITY: PASS")
    print("All Gold tables are stable and validated.")
    print("All key integrity checks passed.")
    print("Audit history is available and healthy.")

else:

    print("\nHEALTHCARE BATCH PIPELINE RELIABILITY: FAIL")
    print("Review the failed checks above before declaring reliability.")

GOLD TABLE RELIABILITY REPORT
+-------------------------+-----------+-------------+--------------+-------------------+------+
|table_name               |actual_rows|expected_rows|null_key_count|duplicate_key_count|status|
+-------------------------+-----------+-------------+--------------+-------------------+------+
|gold_fact_encounters     |61459      |61459        |0             |0                  |PASS  |
|gold_patient_summary     |1163       |1163         |0             |0                  |PASS  |
|gold_condition_summary   |204        |204          |0             |0                  |PASS  |
|gold_procedure_summary   |178        |178          |0             |0                  |PASS  |
|gold_organization_summary|1122       |1122         |0             |0                  |PASS  |
|gold_provider_summary    |1123       |1123         |0             |0                  |PASS  |
|gold_payer_summary       |10         |10           |0             |0                  |PASS  |
+---------